# MetaDiv Core Stats — Detailed Batch CoreValidation
Automatically detects the MetaDiv Builder project directory, reads matched Final Database and Collapse Log files from "output/<MODE>/FINAL_DB", and writes validation outputs to FINAL_DB/validation

In [1]:

from pathlib import Path
import os
import re
import pandas as pd
from IPython.display import display

# ============================================================
# MetaDiv Core Stats — Detailed Batch CoreValidation
#
# Purpose
# -------
# 1. Match every Final Database with its Collapse Log by the
#    complete file-name epithet.
# 2. Count biodiversity units for one selected taxonomic group.
# 3. Show the genera or species collapsed inside that group.
# 4. Export summary, detailed, and issue tables.
#
# Expected project structure
# --------------------------
# MetaDiv_Builder/
# ├── output/
# │   ├── ITS/
# │   │   └── FINAL_DB/
# │   ├── 16S/
# │   │   └── FINAL_DB/
# │   └── CO1/
# │       └── FINAL_DB/
# └── validation scripts or notebooks
#
# The project directory is detected automatically. It can also
# be defined explicitly with the METADIV_PROJECT_DIR environment
# variable.
# ============================================================

MODE = "ITS"  # "ITS", "16S", or "CO1"
TARGET_RANK = "family"
TARGET_NAME = "Amanitaceae"


def resolve_project_directory() -> Path:
    """
    Resolve the MetaDiv Builder project directory.

    The METADIV_PROJECT_DIR environment variable has priority.
    Otherwise, the current working directory and its parents are
    searched for a project containing an ``output`` directory.
    """
    configured_directory = os.environ.get("METADIV_PROJECT_DIR")

    if configured_directory:
        project_directory = Path(
            configured_directory
        ).expanduser().resolve()

        if not project_directory.exists():
            raise FileNotFoundError(
                "METADIV_PROJECT_DIR does not exist:\n"
                f"{project_directory}"
            )

        return project_directory

    current_directory = Path.cwd().resolve()

    for candidate in (
        current_directory,
        *current_directory.parents,
    ):
        if (candidate / "output").is_dir():
            return candidate

    return current_directory


PROJECT_DIR = resolve_project_directory()
INPUT_FOLDER = PROJECT_DIR / "output" / MODE / "FINAL_DB"
VALIDATION_FOLDER = INPUT_FOLDER / "validation"


# ============================================================
# BASIC SETTINGS
# ============================================================

VALID_RANKS = {
    "domain",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
}

TARGET_RANK = TARGET_RANK.strip().casefold()

if TARGET_RANK not in VALID_RANKS:
    raise ValueError(
        f"TARGET_RANK must be one of: {sorted(VALID_RANKS)}"
    )


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_text(value):
    """Normalize taxonomic text for case-insensitive matching."""
    if pd.isna(value):
        return ""
    return str(value).strip().casefold()


def safe_name(value):
    """Create a safe output-file name."""
    return re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value),
    ).strip("_")


def canonical_epithet(path):
    """
    Extract and normalize the full run epithet from a file name.

    Examples
    --------
    Final_Database_species_only_all_eukaryotes_p05.csv
    -> species_only_all_eukaryotes_p05

    Collapse_Log_species_only_all_eukaryotes_p05.txt
    -> species_only_all_eukaryotes_p05
    """
    name = Path(path).stem.strip()

    name = re.sub(
        r"^Final[_\s-]*Database[_\s-]*",
        "",
        name,
        flags=re.IGNORECASE,
    )

    name = re.sub(
        r"^Collapse[_\s-]*Log[_\s-]*",
        "",
        name,
        flags=re.IGNORECASE,
    )

    name = re.sub(
        r"[\s-]+",
        "_",
        name,
    )

    name = re.sub(
        r"_+",
        "_",
        name,
    )

    return name.strip("_").casefold()


def read_final_database(path):
    """Read CSV, TSV, or semicolon-delimited Final Database files."""
    path = Path(path)
    last_error = None

    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            table = pd.read_csv(
                path,
                sep=None,
                engine="python",
                encoding=encoding,
            )

            if table.shape[1] > 1:
                table.columns = [
                    str(column).strip()
                    for column in table.columns
                ]
                return table

        except Exception as error:
            last_error = error

    raise ValueError(
        f"Could not read the Final Database:\n{path}\n\n"
        f"Last error: {last_error}"
    )


def parse_collapse_log(path):
    """
    Read all collapse groups from a MetaDiv Core Stats log.
    """
    path = Path(path)

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    mode_match = re.search(
        r"^\s*MODE:\s*(.+)$",
        text,
        flags=re.MULTILINE,
    )

    subset_match = re.search(
        r"^\s*SUBSET_MODE:\s*(.+)$",
        text,
        flags=re.MULTILINE,
    )

    header_match = re.search(
        r"LOG\s*[—-]\s*([A-Za-z_]+)\s+collapse"
        r"\s*\(p[≥>=]+\s*([0-9.]+)\)",
        text,
        flags=re.IGNORECASE,
    )

    collapse_rank = (
        header_match.group(1).strip().casefold()
        if header_match
        else ""
    )

    threshold = (
        header_match.group(2).strip()
        if header_match
        else ""
    )

    blocks = re.split(
        r"(?=\[Group\s+\d+\]\s+key:)",
        text,
    )

    records = []

    for block in blocks:

        group_match = re.search(
            r"\[Group\s+(\d+)\]\s+key:\s*(.+)",
            block,
        )

        if not group_match:
            continue

        taxonomic_key = group_match.group(2).strip()

        def extract_rank(prefix):
            match = re.search(
                rf"(?:^|_){prefix}:([^_]*)",
                taxonomic_key,
            )
            return match.group(1).strip() if match else ""

        representative_match = re.search(
            r"^\s*Representative:\s*(.+)$",
            block,
            flags=re.MULTILINE,
        )

        length_match = re.search(
            r"^\s*Representative length:\s*(\d+)",
            block,
            flags=re.MULTILINE,
        )

        counts_match = re.search(
            r"^\s*Members:\s*(\d+)\s*\|\s*Removed:\s*(\d+)",
            block,
            flags=re.MULTILINE,
        )

        note_match = re.search(
            r"^\s*Note:\s*(.+)$",
            block,
            flags=re.MULTILINE,
        )

        members = (
            int(counts_match.group(1))
            if counts_match
            else 0
        )

        removed = (
            int(counts_match.group(2))
            if counts_match
            else 0
        )

        records.append({
            "group_number": int(group_match.group(1)),
            "domain": extract_rank("d"),
            "phylum": extract_rank("p"),
            "class": extract_rank("c"),
            "order": extract_rank("o"),
            "family": extract_rank("f"),
            "genus": extract_rank("g"),
            "species": extract_rank("s"),
            "representative": (
                representative_match.group(1).strip()
                if representative_match
                else ""
            ),
            "representative_length": (
                int(length_match.group(1))
                if length_match
                else pd.NA
            ),
            "members_before_collapse": members,
            "units_removed": removed,
            "units_retained_from_group": members - removed,
            "note": (
                note_match.group(1).strip()
                if note_match
                else ""
            ),
            "taxonomic_key": taxonomic_key,
            "log_mode": (
                mode_match.group(1).strip()
                if mode_match
                else ""
            ),
            "log_subset_mode": (
                subset_match.group(1).strip()
                if subset_match
                else ""
            ),
            "log_collapse_rank": collapse_rank,
            "log_threshold": threshold,
        })

    return pd.DataFrame(records)


def detect_subgroup_rank(collapse_log):
    """
    Determine whether the log collapses genus or species units.
    """
    log_rank = normalize_text(
        collapse_log["log_collapse_rank"].iloc[0]
    )

    if log_rank in {"species", "species_only"}:
        return "species"

    if log_rank == "genus":
        return "genus"

    # Fallback based on available taxonomy
    if (
        "species" in collapse_log.columns
        and collapse_log["species"].astype(str).str.strip().ne("").any()
    ):
        return "species"

    return "genus"


# ============================================================
# FIND AND MATCH INPUT FILES
# ============================================================

input_folder = INPUT_FOLDER.resolve()

print(f"Project directory: {PROJECT_DIR}")
print(f"Input folder: {input_folder}")
print(f"Validation folder: {VALIDATION_FOLDER}")

if not input_folder.exists():
    raise FileNotFoundError(
        f"Input folder not found:\n{input_folder}"
    )

VALIDATION_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

final_database_files = sorted([
    path
    for path in input_folder.iterdir()
    if path.is_file()
    and path.suffix.casefold() in {".csv", ".tsv"}
    and re.match(
        r"^Final[_\s-]*Database",
        path.stem,
        flags=re.IGNORECASE,
    )
    and "corevalidation" not in path.stem.casefold()
])

collapse_log_files = sorted([
    path
    for path in input_folder.iterdir()
    if path.is_file()
    and path.suffix.casefold() in {".txt", ".log"}
    and re.match(
        r"^Collapse[_\s-]*Log",
        path.stem,
        flags=re.IGNORECASE,
    )
])

database_map = {
    canonical_epithet(path): path
    for path in final_database_files
}

log_map = {
    canonical_epithet(path): path
    for path in collapse_log_files
}

matched_epithets = sorted(
    set(database_map)
    & set(log_map)
)

unmatched_databases = sorted(
    set(database_map)
    - set(log_map)
)

unmatched_logs = sorted(
    set(log_map)
    - set(database_map)
)

if not matched_epithets:
    raise ValueError(
        "No matching Final Database and Collapse Log pairs were found.\n\n"
        "The complete epithet after Final_Database_ and Collapse_Log_ "
        "must be identical."
    )


# ============================================================
# PROCESS ALL MATCHED PAIRS
# ============================================================

summary_rows = []
detail_tables = []
issue_rows = []

for epithet in matched_epithets:

    final_database_path = database_map[epithet]
    collapse_log_path = log_map[epithet]

    try:
        final_database = read_final_database(
            final_database_path
        )

        collapse_log = parse_collapse_log(
            collapse_log_path
        )

        if TARGET_RANK not in final_database.columns:
            raise KeyError(
                f"Column '{TARGET_RANK}' was not found."
            )

        if collapse_log.empty:
            raise ValueError(
                "No collapse groups were found in the log."
            )

        # All final units belonging to the selected family/group
        final_target = final_database[
            final_database[TARGET_RANK]
            .map(normalize_text)
            .eq(normalize_text(TARGET_NAME))
        ].copy()

        # All collapse groups belonging to the selected family/group
        log_target = collapse_log[
            collapse_log[TARGET_RANK]
            .map(normalize_text)
            .eq(normalize_text(TARGET_NAME))
        ].copy()

        subgroup_rank = detect_subgroup_rank(
            collapse_log
        )

        units_after = int(len(final_target))
        units_removed = int(
            log_target["units_removed"].sum()
        )
        units_before = units_after + units_removed

        summary_rows.append({
            "run_epithet": epithet,
            "final_database": final_database_path.name,
            "collapse_log": collapse_log_path.name,
            "log_mode": collapse_log["log_mode"].iloc[0],
            "log_subset_mode": collapse_log[
                "log_subset_mode"
            ].iloc[0],
            "log_collapse_rank": collapse_log[
                "log_collapse_rank"
            ].iloc[0],
            "log_threshold": collapse_log[
                "log_threshold"
            ].iloc[0],
            "target_rank": TARGET_RANK,
            "target_name": TARGET_NAME,
            "subgroup_rank": subgroup_rank,
            "units_before_collapse": units_before,
            "units_removed": units_removed,
            "units_after_collapse": units_after,
            "number_of_collapse_groups": int(
                len(log_target)
            ),
        })

        if not log_target.empty:
            detail = log_target.copy()

            detail.insert(
                0,
                "run_epithet",
                epithet,
            )

            detail.insert(
                1,
                "final_database",
                final_database_path.name,
            )

            detail.insert(
                2,
                "collapse_log",
                collapse_log_path.name,
            )

            detail.insert(
                3,
                "target_rank",
                TARGET_RANK,
            )

            detail.insert(
                4,
                "target_name",
                TARGET_NAME,
            )

            detail.insert(
                5,
                "subgroup_rank",
                subgroup_rank,
            )

            detail.insert(
                6,
                "subgroup_name",
                detail[subgroup_rank],
            )

            # Count how many rows of each subgroup remain in Final DB
            if subgroup_rank in final_database.columns:
                final_subgroup_counts = (
                    final_target[subgroup_rank]
                    .map(normalize_text)
                    .value_counts()
                    .to_dict()
                )

                detail[
                    "final_database_units_for_subgroup"
                ] = detail[subgroup_rank].map(
                    lambda value: final_subgroup_counts.get(
                        normalize_text(value),
                        0,
                    )
                )
            else:
                detail[
                    "final_database_units_for_subgroup"
                ] = pd.NA

            detail_tables.append(detail)

        print(f"Processed: {epithet}")

    except Exception as error:
        issue_rows.append({
            "run_epithet": epithet,
            "final_database": final_database_path.name,
            "collapse_log": collapse_log_path.name,
            "issue": str(error),
        })

        print(f"Error: {epithet}")


# ============================================================
# REPORT FILES WITHOUT AN EXACT EPITHET MATCH
# ============================================================

for epithet in unmatched_databases:
    issue_rows.append({
        "run_epithet": epithet,
        "final_database": database_map[epithet].name,
        "collapse_log": "",
        "issue": (
            "No Collapse Log with the same complete epithet was found."
        ),
    })

for epithet in unmatched_logs:
    issue_rows.append({
        "run_epithet": epithet,
        "final_database": "",
        "collapse_log": log_map[epithet].name,
        "issue": (
            "No Final Database with the same complete epithet was found."
        ),
    })


# ============================================================
# CREATE OUTPUT TABLES
# ============================================================

summary_table = pd.DataFrame(summary_rows)

if detail_tables:
    detail_table = pd.concat(
        detail_tables,
        ignore_index=True,
    )
else:
    detail_table = pd.DataFrame()

issues_table = pd.DataFrame(issue_rows)


# ============================================================
# SAVE OUTPUTS IN THE VALIDATION FOLDER
# ============================================================

target_label = safe_name(TARGET_NAME)

summary_path = (
    VALIDATION_FOLDER
    / f"CoreValidation_All_Runs_{target_label}_summary.csv"
)

detail_path = (
    VALIDATION_FOLDER
    / f"CoreValidation_All_Runs_{target_label}_details.csv"
)

issues_path = (
    VALIDATION_FOLDER
    / f"CoreValidation_All_Runs_{target_label}_issues.csv"
)

summary_table.to_csv(
    summary_path,
    index=False,
)

detail_table.to_csv(
    detail_path,
    index=False,
)

if not issues_table.empty:
    issues_table.to_csv(
        issues_path,
        index=False,
    )


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\nDetailed Batch CoreValidation completed.")
print(f"Matched pairs: {len(matched_epithets)}")
print(f"Successful runs: {len(summary_table)}")
print(f"Target: {TARGET_NAME} ({TARGET_RANK})")

print(f"\nSummary saved to:\n{summary_path}")
print(f"\nDetails saved to:\n{detail_path}")

display(summary_table)

if not detail_table.empty:
    print("\nCollapsed genera or species inside the target group:")
    display(detail_table)

if not issues_table.empty:
    print("\nFiles without an exact epithet match or processing errors:")
    display(issues_table)
    print(f"\nIssues saved to:\n{issues_path}")


Project directory: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC
Input folder: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB
Validation folder: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation
Processed: genus_all_eukaryotes_p1

Detailed Batch CoreValidation completed.
Matched pairs: 1
Successful runs: 1
Target: Amanitaceae (family)

Summary saved to:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\CoreValidation_All_Runs_Amanitaceae_summary.csv

Details saved to:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\CoreValidation_All_Runs_Amanitaceae_details.csv


,run_epithet,final_database,collapse_log,log_mode,log_subset_mode,log_collapse_rank,log_threshold,target_rank,target_name,subgroup_rank,units_before_collapse,units_removed,units_after_collapse,number_of_collapse_groups
0,genus_all_eukaryotes_p1,Final_Database_genus_all_eukaryotes_p1.csv,Collapse_Log_genus_all_eukaryotes_p1.txt,ITS,all_eukaryotes,genus,1.0,family,Amanitaceae,genus,181,111,70,4



Collapsed genera or species inside the target group:


,run_epithet,final_database,collapse_log,target_rank,target_name,subgroup_rank,subgroup_name,group_number,domain,phylum,...,members_before_collapse,units_removed,units_retained_from_group,note,taxonomic_key,log_mode,log_subset_mode,log_collapse_rank,log_threshold,final_database_units_for_subgroup
0,genus_all_eukaryotes_p1,Final_Database_genus_all_eukaryotes_p1.csv,Collapse_Log_genus_all_eukaryotes_p1.txt,family,Amanitaceae,genus,Limacella,100,Fungi,Basidiomycota,...,5,0,5,skip_collapse_incomplete_tax_genus,d:Fungi_p:Basidiomycota_c:Agaricomycetes_o:Aga...,ITS,all_eukaryotes,genus,1.0,10
1,genus_all_eukaryotes_p1,Final_Database_genus_all_eukaryotes_p1.csv,Collapse_Log_genus_all_eukaryotes_p1.txt,family,Amanitaceae,genus,Amanita,119,Fungi,Basidiomycota,...,110,109,1,collapsed_with_complete_tax_genus,d:Fungi_p:Basidiomycota_c:Agaricomycetes_o:Aga...,ITS,all_eukaryotes,genus,1.0,49
2,genus_all_eukaryotes_p1,Final_Database_genus_all_eukaryotes_p1.csv,Collapse_Log_genus_all_eukaryotes_p1.txt,family,Amanitaceae,genus,Saproamanita,855,Fungi,Basidiomycota,...,3,2,1,collapsed_with_complete_tax_genus,d:Fungi_p:Basidiomycota_c:Agaricomycetes_o:Aga...,ITS,all_eukaryotes,genus,1.0,3
3,genus_all_eukaryotes_p1,Final_Database_genus_all_eukaryotes_p1.csv,Collapse_Log_genus_all_eukaryotes_p1.txt,family,Amanitaceae,genus,Zhuliangomyces,1373,Fungi,Basidiomycota,...,2,0,2,skip_collapse_incomplete_tax_genus,d:Fungi_p:Basidiomycota_c:Agaricomycetes_o:Aga...,ITS,all_eukaryotes,genus,1.0,6



Files without an exact epithet match or processing errors:


,run_epithet,final_database,collapse_log,issue
0,genus_only_fungi_p1,Final_Database_genus_only_fungi_p1.csv,,No Collapse Log with the same complete epithet...



Issues saved to:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\CoreValidation_All_Runs_Amanitaceae_issues.csv


# MetaDiv Builder — Longest Representative Sequence Validation
This notebook automatically resolves the MetaDiv Builder project directory and selects the corresponding pre-collapse tables, Final Database, and Collapse Log from the configured marker, subset, collapse strategy, and confidence threshold.

Validation files are written to output/<MODE>/FINAL_DB/validation/.

In [2]:
from pathlib import Path
import os
import re
import pandas as pd
from IPython.display import display


# ============================================================
# MetaDiv Builder
# Longest Representative Sequence Validation
#
# Purpose
# -------
# Validate whether the representative sequence retained for each
# harmonized biodiversity unit corresponds to a sequence with
# the maximum length among all original sequences contributing
# to the taxonomic collapse group.
#
# Input structure
# ---------------
# 1. PRE_COLLAPSE_FOLDER
#    Folder containing multiple files ending in:
#
#        _concatenated.csv
#
#    Expected main columns:
#
#        OTU_XX
#        Original_ID
#        sintax_taxonomy
#        sequence
#        abundance columns...
#
# 2. FINAL_DATABASE_FILE
#    One harmonized Final Database CSV.
#
# 3. COLLAPSE_LOG_FILE
#    One Collapse Log TXT corresponding to the Final Database.
#
# Main validation
# ---------------
# For every selected collapse group, the script:
#
# - reconstructs the original group from all concatenated tables;
# - preserves the dataset of origin;
# - calculates sequence length directly from the sequence;
# - identifies the maximum sequence length;
# - verifies whether the log representative is one of the
#   longest available sequences;
# - verifies whether the representative remains in the Final DB;
# - calculates the sequence length stored in the Final Database;
# - identifies the shortest discarded candidate sequence;
# - calculates the difference in base pairs between the retained
#   representative and the shortest discarded candidate;
# - counts how many candidates each dataset contributed;
# - exports detailed CSV validation reports.
# ============================================================


# ============================================================
# 1. USER SETTINGS AND PROJECT PATHS
# ============================================================

MODE = "ITS"  # "ITS", "16S", or "CO1"

# Valid subset modes depend on MODE:
# ITS: "only_fungi" or "all_eukaryotes"
# 16S: "only_bacteria" or "all_prokaryotes"
# CO1: "only_metazoa" or "all_eukaryotes"
SUBSET_MODE = "all_eukaryotes"

COLLAPSE_STRATEGY = "genus"  # "species_only", "genus", or "all"
P_VALUE_THRESHOLD = 1.00

# Main taxonomic target
TARGET_RANK = "family"
TARGET_NAME = "Amanitaceae"

# Optional second restriction
SUBTARGET_RANK = "genus"
SUBTARGET_NAME = "Amanita"

# To validate the complete main target:
# SUBTARGET_RANK = None
# SUBTARGET_NAME = None


def resolve_project_directory() -> Path:
    """
    Resolve the MetaDiv Builder project directory.

    The METADIV_PROJECT_DIR environment variable has priority.
    Otherwise, the current working directory and its parent
    directories are searched for a project containing an
    ``output`` directory.
    """
    configured_directory = os.environ.get("METADIV_PROJECT_DIR")

    if configured_directory:
        project_directory = Path(
            configured_directory
        ).expanduser().resolve()

        if not project_directory.exists():
            raise FileNotFoundError(
                "METADIV_PROJECT_DIR does not exist:\n"
                f"{project_directory}"
            )

        return project_directory

    current_directory = Path.cwd().resolve()

    for candidate in (
        current_directory,
        *current_directory.parents,
    ):
        if (candidate / "output").is_dir():
            return candidate

    return current_directory


def threshold_token(value) -> str:
    """
    Convert a numeric threshold to the MetaDiv file-name token.

    Examples
    --------
    1.00 -> p1
    0.80 -> p08
    0.50 -> p05
    0.00 -> p0
    """
    numeric_value = float(value)

    if not 0.0 <= numeric_value <= 1.0:
        raise ValueError(
            "P_VALUE_THRESHOLD must be between 0.00 and 1.00."
        )

    formatted = f"{numeric_value:.2f}".rstrip("0").rstrip(".")

    if formatted.startswith("0."):
        formatted = formatted.replace("0.", "0", 1)

    return f"p{formatted}"


def build_run_epithet() -> str:
    """
    Build the complete MetaDiv run epithet used in output files.
    """
    return "_".join(
        [
            COLLAPSE_STRATEGY,
            SUBSET_MODE,
            threshold_token(P_VALUE_THRESHOLD),
        ]
    )


PROJECT_DIR = resolve_project_directory()
MARKER_OUTPUT_FOLDER = PROJECT_DIR / "output" / MODE
PRE_COLLAPSE_FOLDER = MARKER_OUTPUT_FOLDER / "concatenated_tables"
FINAL_DB_FOLDER = MARKER_OUTPUT_FOLDER / "FINAL_DB"
OUTPUT_FOLDER = FINAL_DB_FOLDER / "validation"

RUN_EPITHET = build_run_epithet()

FINAL_DATABASE_FILE = (
    FINAL_DB_FOLDER
    / f"Final_Database_{RUN_EPITHET}.csv"
)

COLLAPSE_LOG_FILE = (
    FINAL_DB_FOLDER
    / f"Collapse_Log_{RUN_EPITHET}.txt"
)

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. CONSTANTS AND COLUMN ALIASES
# ============================================================

VALID_RANKS = {
    "domain",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
}

RANK_PREFIXES = {
    "domain": "d",
    "phylum": "p",
    "class": "c",
    "order": "o",
    "family": "f",
    "genus": "g",
    "species": "s",
}

ID_CANDIDATES = [
    "OTU_XX",
    "otu_xx",
    "OTU_ID",
    "otu_id",
    "OTU",
    "otu",
    "FeatureID",
    "feature_id",
    "featureid",
    "feature",
    "ASV",
    "asv",
    "zOTU",
    "zotu",
    "SequenceID",
    "sequence_id",
    "ID",
    "id",
]

ORIGINAL_ID_CANDIDATES = [
    "Original_ID",
    "original_id",
    "OriginalID",
    "originalid",
]

SEQUENCE_CANDIDATES = [
    "sequence",
    "Sequence",
    "SEQUENCE",
    "representative_sequence",
    "Representative_Sequence",
    "Representative_sequence",
    "dna_sequence",
    "DNA_sequence",
    "nucleotide_sequence",
    "Nucleotide_sequence",
]

TAXONOMY_CANDIDATES = [
    "sintax_taxonomy",
    "SINTAX_taxonomy",
    "Sintax_taxonomy",
    "taxonomy",
    "Taxonomy",
    "taxonomic_assignment",
]

LENGTH_CANDIDATES = [
    "sequence_length",
    "Sequence_length",
    "Sequence_Length",
    "sequence_lenght",
    "length",
    "Length",
]


# ============================================================
# 3. VALIDATE USER SETTINGS
# ============================================================

VALID_SUBSET_MODES = {
    "ITS": {"only_fungi", "all_eukaryotes"},
    "16S": {"only_bacteria", "all_prokaryotes"},
    "CO1": {"only_metazoa", "all_eukaryotes"},
}

VALID_COLLAPSE_STRATEGIES = {
    "species_only",
    "genus",
    "all",
}

MODE = MODE.strip().upper()
SUBSET_MODE = SUBSET_MODE.strip().casefold()
COLLAPSE_STRATEGY = COLLAPSE_STRATEGY.strip().casefold()

if MODE not in VALID_SUBSET_MODES:
    raise ValueError(
        "MODE must be one of: ITS, 16S, or CO1."
    )

if SUBSET_MODE not in VALID_SUBSET_MODES[MODE]:
    raise ValueError(
        f"SUBSET_MODE '{SUBSET_MODE}' is not valid for MODE "
        f"'{MODE}'. Accepted values: "
        f"{sorted(VALID_SUBSET_MODES[MODE])}"
    )

if COLLAPSE_STRATEGY not in VALID_COLLAPSE_STRATEGIES:
    raise ValueError(
        "COLLAPSE_STRATEGY must be one of: "
        f"{sorted(VALID_COLLAPSE_STRATEGIES)}"
    )

TARGET_RANK = TARGET_RANK.strip().casefold()

if TARGET_RANK not in VALID_RANKS:
    raise ValueError(
        f"TARGET_RANK must be one of: {sorted(VALID_RANKS)}"
    )

if SUBTARGET_RANK is not None:
    SUBTARGET_RANK = SUBTARGET_RANK.strip().casefold()

    if SUBTARGET_RANK not in VALID_RANKS:
        raise ValueError(
            f"SUBTARGET_RANK must be one of: "
            f"{sorted(VALID_RANKS)}"
        )

if SUBTARGET_RANK is None and SUBTARGET_NAME is not None:
    raise ValueError(
        "SUBTARGET_NAME must be None when "
        "SUBTARGET_RANK is None."
    )

if SUBTARGET_RANK is not None and SUBTARGET_NAME is None:
    raise ValueError(
        "SUBTARGET_NAME must be defined when "
        "SUBTARGET_RANK is used."
    )

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. GENERAL HELPER FUNCTIONS
# ============================================================

def normalize_text(value):
    """
    Normalize text for case-insensitive comparisons.
    """
    if pd.isna(value):
        return ""

    return str(value).strip().casefold()


def safe_name(value):
    """
    Convert text into a safe file-name component.
    """
    return re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value),
    ).strip("_")


def normalize_sequence(value):
    """
    Normalize a nucleotide sequence before calculating length.

    Removes:
    - spaces;
    - line breaks;
    - tabs;
    - gap symbols '-' and '.'.
    """
    if pd.isna(value):
        return ""

    sequence = str(value).strip()

    sequence = re.sub(
        r"\s+",
        "",
        sequence,
    )

    sequence = sequence.replace("-", "")
    sequence = sequence.replace(".", "")

    return sequence.upper()


def read_database(path):
    """
    Read a delimited table.

    First attempts automatic delimiter detection using the Python
    engine. If that fails, common delimiters are tested with the
    C engine.

    Important:
    low_memory is not used with engine='python' because pandas
    does not support that combination.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Input file not found:\n{path}"
        )

    last_error = None

    encodings = [
        "utf-8",
        "utf-8-sig",
        "latin-1",
    ]

    # Automatic delimiter detection
    for encoding in encodings:
        try:
            table = pd.read_csv(
                path,
                sep=None,
                engine="python",
                encoding=encoding,
            )

            if table.shape[1] > 1:
                table.columns = [
                    str(column).strip()
                    for column in table.columns
                ]

                return table

        except Exception as error:
            last_error = error

    # Explicit delimiter fallback
    delimiters = [
        "\t",
        ",",
        ";",
        "|",
    ]

    for encoding in encodings:
        for delimiter in delimiters:
            try:
                table = pd.read_csv(
                    path,
                    sep=delimiter,
                    engine="c",
                    encoding=encoding,
                    low_memory=False,
                )

                if table.shape[1] > 1:
                    table.columns = [
                        str(column).strip()
                        for column in table.columns
                    ]

                    return table

            except Exception as error:
                last_error = error

    raise ValueError(
        f"Could not read the table:\n{path}\n\n"
        f"Last error: {last_error}"
    )


def detect_column(
    table,
    candidates,
    label,
):
    """
    Detect a required column using accepted aliases.
    """
    normalized_columns = {
        normalize_text(column): column
        for column in table.columns
    }

    for candidate in candidates:
        normalized_candidate = normalize_text(
            candidate
        )

        if normalized_candidate in normalized_columns:
            return normalized_columns[
                normalized_candidate
            ]

    raise KeyError(
        f"Could not detect the {label} column.\n"
        f"Accepted names: {candidates}\n"
        f"Available columns: {list(table.columns)}"
    )


def detect_optional_column(
    table,
    candidates,
):
    """
    Detect an optional column.

    Returns None if no matching column exists.
    """
    normalized_columns = {
        normalize_text(column): column
        for column in table.columns
    }

    for candidate in candidates:
        normalized_candidate = normalize_text(
            candidate
        )

        if normalized_candidate in normalized_columns:
            return normalized_columns[
                normalized_candidate
            ]

    return None


def standardize_taxonomic_columns(table):
    """
    Rename existing taxonomic columns to lowercase canonical
    names.
    """
    rename_map = {}

    for column in table.columns:
        normalized = normalize_text(column)

        if normalized in VALID_RANKS:
            rename_map[column] = normalized

    return table.rename(
        columns=rename_map
    )


def normalize_taxon_for_rank(
    value,
    rank,
):
    """
    Normalize taxonomic values for rank-specific matching.

    At species level:
        Amanita_muscaria
        Amanita muscaria
        muscaria

    are reduced to:
        muscaria
    """
    normalized = normalize_text(value)

    normalized = normalized.replace(
        " ",
        "_",
    )

    if rank == "species" and "_" in normalized:
        normalized = normalized.split("_")[-1]

    return normalized


# ============================================================
# 5. SINTAX TAXONOMY PARSER
# ============================================================

def parse_sintax_taxonomy(value):
    """
    Parse a SINTAX taxonomy string into canonical ranks.

    Supported examples
    ------------------
    d:Eukaryota,p:Fungi,c:Agaricomycetes,o:Agaricales,
    f:Amanitaceae,g:Amanita,s:Amanita_muscaria

    d:Eukaryota(1.000),p:Fungi(0.999),
    c:Agaricomycetes(0.950),...

    Confidence values enclosed in parentheses are removed.
    """
    parsed = {
        "domain": "",
        "phylum": "",
        "class": "",
        "order": "",
        "family": "",
        "genus": "",
        "species": "",
    }

    if pd.isna(value):
        return parsed

    taxonomy = str(value).strip()

    if taxonomy == "":
        return parsed

    prefix_to_rank = {
        "d": "domain",
        "k": "domain",
        "p": "phylum",
        "c": "class",
        "o": "order",
        "f": "family",
        "g": "genus",
        "s": "species",
    }

    components = re.split(
        r"[,;]",
        taxonomy,
    )

    for component in components:

        component = component.strip()

        match = re.match(
            r"^([dkpcofgs])\s*:\s*(.+)$",
            component,
            flags=re.IGNORECASE,
        )

        if not match:
            continue

        prefix = match.group(1).casefold()
        taxon = match.group(2).strip()

        # Remove confidence values such as:
        # Amanita(0.983)
        taxon = re.sub(
            r"\s*\([^)]*\)\s*$",
            "",
            taxon,
        ).strip()

        rank = prefix_to_rank.get(prefix)

        if rank is not None:
            parsed[rank] = taxon

    return parsed


def expand_sintax_taxonomy(
    table,
    taxonomy_column,
):
    """
    Expand sintax_taxonomy into separate taxonomic columns.

    Existing non-empty taxonomic columns are preserved.
    """
    parsed_taxonomy = (
        table[taxonomy_column]
        .map(parse_sintax_taxonomy)
        .apply(pd.Series)
    )

    for rank in [
        "domain",
        "phylum",
        "class",
        "order",
        "family",
        "genus",
        "species",
    ]:

        if rank not in table.columns:
            table[rank] = parsed_taxonomy[rank]

        else:
            existing_values = (
                table[rank]
                .fillna("")
                .astype(str)
                .str.strip()
            )

            missing_mask = existing_values.eq("")

            table.loc[
                missing_mask,
                rank,
            ] = parsed_taxonomy.loc[
                missing_mask,
                rank,
            ]

    return table


# ============================================================
# 6. COLLAPSE LOG PARSER
# ============================================================

def extract_taxonomic_rank(
    taxonomic_key,
    prefix,
):
    """
    Extract one rank from a MetaDiv taxonomic key.

    Example
    -------
    d:Eukaryota_p:Fungi_c:Agaricomycetes_
    o:Agaricales_f:Amanitaceae_g:Amanita_s:muscaria
    """
    match = re.search(
        rf"(?:^|_){re.escape(prefix)}:([^_]*)",
        str(taxonomic_key),
    )

    if match:
        return match.group(1).strip()

    return ""


def parse_collapse_log(path):
    """
    Parse all collapse groups from a MetaDiv Collapse Log.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Collapse Log not found:\n{path}"
        )

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    mode_match = re.search(
        r"^\s*MODE:\s*(.+)$",
        text,
        flags=re.MULTILINE,
    )

    subset_match = re.search(
        r"^\s*SUBSET_MODE:\s*(.+)$",
        text,
        flags=re.MULTILINE,
    )

    header_match = re.search(
        r"LOG\s*[—-]\s*([A-Za-z_]+)\s+collapse"
        r"\s*\(p[≥>=]+\s*([0-9.]+)\)",
        text,
        flags=re.IGNORECASE,
    )

    collapse_rank = (
        header_match.group(1)
        .strip()
        .casefold()
        if header_match
        else ""
    )

    threshold = (
        header_match.group(2).strip()
        if header_match
        else ""
    )

    blocks = re.split(
        r"(?=\[Group\s+\d+\]\s+key:)",
        text,
    )

    records = []

    for block in blocks:

        group_match = re.search(
            r"\[Group\s+(\d+)\]\s+key:\s*(.+)",
            block,
        )

        if not group_match:
            continue

        group_number = int(
            group_match.group(1)
        )

        taxonomic_key = (
            group_match.group(2).strip()
        )

        representative_match = re.search(
            r"^\s*Representative:\s*(.+)$",
            block,
            flags=re.MULTILINE,
        )

        representative_length_match = re.search(
            r"^\s*Representative length:\s*(\d+)",
            block,
            flags=re.MULTILINE,
        )

        counts_match = re.search(
            r"^\s*Members:\s*(\d+)"
            r"\s*\|\s*Removed:\s*(\d+)",
            block,
            flags=re.MULTILINE,
        )

        note_match = re.search(
            r"^\s*Note:\s*(.+)$",
            block,
            flags=re.MULTILINE,
        )

        record = {
            "group_number": group_number,
            "taxonomic_key": taxonomic_key,
            "representative": (
                representative_match
                .group(1)
                .strip()
                if representative_match
                else ""
            ),
            "reported_representative_length": (
                int(
                    representative_length_match
                    .group(1)
                )
                if representative_length_match
                else pd.NA
            ),
            "reported_members": (
                int(counts_match.group(1))
                if counts_match
                else pd.NA
            ),
            "reported_removed": (
                int(counts_match.group(2))
                if counts_match
                else pd.NA
            ),
            "note": (
                note_match.group(1).strip()
                if note_match
                else ""
            ),
            "log_mode": (
                mode_match.group(1).strip()
                if mode_match
                else ""
            ),
            "log_subset_mode": (
                subset_match.group(1).strip()
                if subset_match
                else ""
            ),
            "log_collapse_rank": collapse_rank,
            "log_threshold": threshold,
        }

        for rank, prefix in RANK_PREFIXES.items():
            record[rank] = extract_taxonomic_rank(
                taxonomic_key,
                prefix,
            )

        records.append(record)

    return pd.DataFrame(records)


# ============================================================
# 7. READ ALL PRE-COLLAPSE DATASETS
# ============================================================

def read_precollapse_folder(folder):
    """
    Read every file ending in '_concatenated.csv'.

    Expected main columns:
    - OTU_XX
    - Original_ID
    - sintax_taxonomy
    - sequence

    Sample abundance columns are retained but are not used for
    longest-sequence validation.
    """
    folder = Path(folder)

    if not folder.exists():
        raise FileNotFoundError(
            f"Pre-collapse folder not found:\n{folder}"
        )

    input_files = sorted([
        path
        for path in folder.iterdir()
        if path.is_file()
        and path.name.casefold().endswith(
            "_concatenated.csv"
        )
        and not path.name.startswith("~$")
    ])

    if not input_files:
        raise ValueError(
            "No files ending in '_concatenated.csv' "
            "were found in:\n"
            f"{folder}"
        )

    combined_tables = []
    summary_rows = []
    issue_rows = []

    for input_path in input_files:

        source_dataset_name = re.sub(
            r"_concatenated$",
            "",
            input_path.stem,
            flags=re.IGNORECASE,
        )

        try:
            table = read_database(
                input_path
            )

            table = standardize_taxonomic_columns(
                table
            )

            feature_id_column = detect_column(
                table,
                ID_CANDIDATES,
                (
                    "feature identifier in "
                    f"{input_path.name}"
                ),
            )

            sequence_column = detect_column(
                table,
                SEQUENCE_CANDIDATES,
                (
                    "sequence in "
                    f"{input_path.name}"
                ),
            )

            taxonomy_column = detect_column(
                table,
                TAXONOMY_CANDIDATES,
                (
                    "SINTAX taxonomy in "
                    f"{input_path.name}"
                ),
            )

            original_id_column = detect_optional_column(
                table,
                ORIGINAL_ID_CANDIDATES,
            )

            reported_length_column = detect_optional_column(
                table,
                LENGTH_CANDIDATES,
            )

            # Expand SINTAX taxonomy before renaming
            table = expand_sintax_taxonomy(
                table,
                taxonomy_column,
            )

            rename_map = {
                feature_id_column: "_feature_id",
                sequence_column: "_sequence",
                taxonomy_column: "_sintax_taxonomy",
            }

            if original_id_column is not None:
                rename_map[
                    original_id_column
                ] = "_original_id"

            if (
                reported_length_column is not None
                and reported_length_column
                not in rename_map
            ):
                rename_map[
                    reported_length_column
                ] = "_reported_sequence_length"

            table = table.rename(
                columns=rename_map
            )

            table.insert(
                0,
                "source_dataset",
                source_dataset_name,
            )

            table.insert(
                1,
                "source_file",
                input_path.name,
            )

            table["_feature_id"] = (
                table["_feature_id"]
                .fillna("")
                .astype(str)
                .str.strip()
            )

            if "_original_id" not in table.columns:
                table["_original_id"] = ""

            else:
                table["_original_id"] = (
                    table["_original_id"]
                    .fillna("")
                    .astype(str)
                    .str.strip()
                )

            table["_normalized_sequence"] = (
                table["_sequence"]
                .map(normalize_sequence)
            )

            table["_calculated_length"] = (
                table["_normalized_sequence"]
                .str.len()
            )

            table["_dataset_feature_key"] = (
                table["source_dataset"]
                .astype(str)
                + "::"
                + table["_feature_id"]
                .astype(str)
            )

            empty_feature_ids = int(
                table["_feature_id"]
                .eq("")
                .sum()
            )

            empty_sequences = int(
                table["_calculated_length"]
                .eq(0)
                .sum()
            )

            duplicate_dataset_keys = int(
                table["_dataset_feature_key"]
                .duplicated()
                .sum()
            )

            target_rows = int(
                table[TARGET_RANK]
                .map(
                    lambda value: normalize_taxon_for_rank(
                        value,
                        TARGET_RANK,
                    )
                )
                .eq(
                    normalize_taxon_for_rank(
                        TARGET_NAME,
                        TARGET_RANK,
                    )
                )
                .sum()
            )

            if (
                SUBTARGET_RANK is not None
                and SUBTARGET_NAME is not None
            ):
                subtarget_rows = int(
                    table[SUBTARGET_RANK]
                    .map(
                        lambda value: normalize_taxon_for_rank(
                            value,
                            SUBTARGET_RANK,
                        )
                    )
                    .eq(
                        normalize_taxon_for_rank(
                            SUBTARGET_NAME,
                            SUBTARGET_RANK,
                        )
                    )
                    .sum()
                )
            else:
                subtarget_rows = pd.NA

            valid_lengths = table.loc[
                table["_calculated_length"].gt(0),
                "_calculated_length",
            ]

            combined_tables.append(
                table
            )

            summary_rows.append({
                "source_dataset": source_dataset_name,
                "source_file": input_path.name,
                "rows_loaded": len(table),
                "unique_feature_ids": (
                    table["_feature_id"].nunique()
                ),
                "empty_feature_ids": empty_feature_ids,
                "unique_dataset_feature_keys": (
                    table["_dataset_feature_key"]
                    .nunique()
                ),
                "duplicate_dataset_feature_keys": (
                    duplicate_dataset_keys
                ),
                "sequences_with_length_gt_zero": int(
                    table["_calculated_length"]
                    .gt(0)
                    .sum()
                ),
                "empty_sequences": empty_sequences,
                "minimum_sequence_length": (
                    valid_lengths.min()
                    if not valid_lengths.empty
                    else pd.NA
                ),
                "maximum_sequence_length": (
                    valid_lengths.max()
                    if not valid_lengths.empty
                    else pd.NA
                ),
                "target_rows": target_rows,
                "subtarget_rows": subtarget_rows,
                "status": "LOADED",
            })

            print(
                f"Loaded: {input_path.name} | "
                f"rows: {len(table):,} | "
                f"target rows: {target_rows:,}"
            )

        except Exception as error:

            issue_rows.append({
                "source_dataset": source_dataset_name,
                "source_file": input_path.name,
                "issue": str(error),
            })

            summary_rows.append({
                "source_dataset": source_dataset_name,
                "source_file": input_path.name,
                "rows_loaded": 0,
                "unique_feature_ids": 0,
                "empty_feature_ids": 0,
                "unique_dataset_feature_keys": 0,
                "duplicate_dataset_feature_keys": 0,
                "sequences_with_length_gt_zero": 0,
                "empty_sequences": 0,
                "minimum_sequence_length": pd.NA,
                "maximum_sequence_length": pd.NA,
                "target_rows": 0,
                "subtarget_rows": 0,
                "status": "ERROR",
            })

            print(
                f"Error reading {input_path.name}: "
                f"{error}"
            )

    if not combined_tables:

        issues_text = "\n".join(
            f"- {row['source_file']}: {row['issue']}"
            for row in issue_rows
        )

        raise ValueError(
            "None of the original pre-collapse datasets "
            "could be loaded.\n\n"
            f"Detected issues:\n{issues_text}"
        )

    combined_database = pd.concat(
        combined_tables,
        ignore_index=True,
        sort=False,
    )

    source_summary = pd.DataFrame(
        summary_rows
    )

    source_issues = pd.DataFrame(
        issue_rows
    )

    return (
        combined_database,
        source_summary,
        source_issues,
    )


# ============================================================
# 8. SELECT TAXONOMIC GROUPS
# ============================================================

def subset_taxonomic_group(
    table,
    rank,
    name,
):
    """
    Select rows belonging to one taxonomic group.
    """
    if rank not in table.columns:
        raise KeyError(
            f"Column '{rank}' was not found."
        )

    return table[
        table[rank]
        .map(
            lambda value: normalize_taxon_for_rank(
                value,
                rank,
            )
        )
        .eq(
            normalize_taxon_for_rank(
                name,
                rank,
            )
        )
    ].copy()


def reconstruct_collapse_group(
    source_table,
    log_row,
):
    """
    Reconstruct a collapse group from all original datasets.

    Every non-empty taxonomic rank present in both the Collapse
    Log and the source table is used.
    """
    group = source_table.copy()
    matched_ranks = []

    ordered_ranks = [
        "domain",
        "phylum",
        "class",
        "order",
        "family",
        "genus",
        "species",
    ]

    for rank in ordered_ranks:

        if rank not in source_table.columns:
            continue

        log_value = log_row.get(
            rank,
            "",
        )

        if normalize_text(log_value) == "":
            continue

        normalized_log_value = (
            normalize_taxon_for_rank(
                log_value,
                rank,
            )
        )

        group = group[
            group[rank]
            .map(
                lambda value: normalize_taxon_for_rank(
                    value,
                    rank,
                )
            )
            .eq(normalized_log_value)
        ]

        matched_ranks.append(rank)

    return group.copy(), matched_ranks


# ============================================================
# 9. READ INPUT FILES
# ============================================================

print(f"Project directory: {PROJECT_DIR}")
print(f"Marker mode: {MODE}")
print(f"Subset mode: {SUBSET_MODE}")
print(f"Run epithet: {RUN_EPITHET}")
print(f"Pre-collapse folder: {PRE_COLLAPSE_FOLDER}")
print(f"Final Database: {FINAL_DATABASE_FILE}")
print(f"Collapse Log: {COLLAPSE_LOG_FILE}")
print(f"Validation output folder: {OUTPUT_FOLDER}")

required_paths = {
    "Pre-collapse folder": PRE_COLLAPSE_FOLDER,
    "Final Database": FINAL_DATABASE_FILE,
    "Collapse Log": COLLAPSE_LOG_FILE,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.exists()
]

if missing_paths:
    available_databases = sorted(
        path.name
        for path in FINAL_DB_FOLDER.glob("Final_Database_*.csv")
    ) if FINAL_DB_FOLDER.exists() else []

    message = (
        "The selected run could not be located.\n\n"
        + "\n".join(missing_paths)
    )

    if available_databases:
        message += (
            "\n\nAvailable Final Database files:\n- "
            + "\n- ".join(available_databases)
        )

    raise FileNotFoundError(message)

print(
    "\nReading original pre-collapse datasets..."
)

(
    precollapse_database,
    source_dataset_summary,
    source_dataset_issues,
) = read_precollapse_folder(
    PRE_COLLAPSE_FOLDER
)

print(
    "\nReading Final Database..."
)

final_database = read_database(
    FINAL_DATABASE_FILE
)

final_database = standardize_taxonomic_columns(
    final_database
)

print(
    "\nReading Collapse Log..."
)

collapse_log = parse_collapse_log(
    COLLAPSE_LOG_FILE
)

if collapse_log.empty:
    raise ValueError(
        "No collapse groups were detected in the "
        "Collapse Log."
    )

print(
    "\nOriginal pre-collapse datasets loaded:"
)

display(
    source_dataset_summary
)

print(
    "\nTotal original rows loaded: "
    f"{len(precollapse_database):,}"
)

print(
    "Total original datasets loaded: "
    f"{precollapse_database['source_dataset'].nunique():,}"
)


# ============================================================
# 10. PREPARE FINAL DATABASE
# ============================================================

final_id_column = detect_column(
    final_database,
    ID_CANDIDATES,
    "Final Database feature identifier",
)

final_sequence_column = detect_optional_column(
    final_database,
    SEQUENCE_CANDIDATES,
)

final_length_column = detect_optional_column(
    final_database,
    LENGTH_CANDIDATES,
)

final_database[final_id_column] = (
    final_database[final_id_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

if final_sequence_column is not None:

    final_database["_normalized_sequence"] = (
        final_database[final_sequence_column]
        .map(normalize_sequence)
    )

    final_database["_calculated_length"] = (
        final_database["_normalized_sequence"]
        .str.len()
    )

elif final_length_column is not None:

    final_database["_normalized_sequence"] = ""

    final_database["_calculated_length"] = (
        pd.to_numeric(
            final_database[final_length_column],
            errors="coerce",
        )
    )

else:

    final_database["_normalized_sequence"] = ""
    final_database["_calculated_length"] = pd.NA


# ============================================================
# 11. SELECT TARGET COLLAPSE GROUPS
# ============================================================

target_logs = subset_taxonomic_group(
    collapse_log,
    TARGET_RANK,
    TARGET_NAME,
)

if (
    SUBTARGET_RANK is not None
    and SUBTARGET_NAME is not None
):

    target_logs = subset_taxonomic_group(
        target_logs,
        SUBTARGET_RANK,
        SUBTARGET_NAME,
    )

if target_logs.empty:

    target_description = (
        f"{TARGET_RANK}={TARGET_NAME}"
    )

    if SUBTARGET_RANK is not None:
        target_description += (
            f", {SUBTARGET_RANK}="
            f"{SUBTARGET_NAME}"
        )

    raise ValueError(
        "No collapse groups matched the selected "
        "taxonomic target:\n"
        f"{target_description}"
    )

print(
    "\nSelected collapse groups: "
    f"{len(target_logs):,}"
)


# ============================================================
# 12. VALIDATE EACH COLLAPSE GROUP
# ============================================================

validation_rows = []
member_tables = []
dataset_count_tables = []
shortest_discarded_tables = []

for _, log_row in target_logs.iterrows():

    group_number = int(
        log_row["group_number"]
    )

    representative = str(
        log_row["representative"]
    ).strip()

    group_members, matched_ranks = (
        reconstruct_collapse_group(
            precollapse_database,
            log_row,
        )
    )

    # Locate the retained representative directly in the
    # Final Database and calculate its stored sequence length.
    final_representative_rows = final_database[
        final_database[final_id_column]
        .eq(representative)
    ].copy()

    representative_found_in_final = (
        not final_representative_rows.empty
    )

    if representative_found_in_final:

        valid_final_lengths = pd.to_numeric(
            final_representative_rows[
                "_calculated_length"
            ],
            errors="coerce",
        ).dropna()

        final_database_sequence_length = (
            int(valid_final_lengths.iloc[0])
            if not valid_final_lengths.empty
            else pd.NA
        )

    else:

        final_database_sequence_length = pd.NA

    # --------------------------------------------------------
    # Group could not be reconstructed
    # --------------------------------------------------------

    if group_members.empty:

        validation_rows.append({
            "group_number": group_number,
            "domain": log_row.get("domain", ""),
            "phylum": log_row.get("phylum", ""),
            "class": log_row.get("class", ""),
            "order": log_row.get("order", ""),
            "family": log_row.get("family", ""),
            "genus": log_row.get("genus", ""),
            "species": log_row.get("species", ""),
            "taxonomic_key": log_row["taxonomic_key"],
            "representative": representative,
            "status": "GROUP_NOT_RECONSTRUCTED",
            "matched_taxonomic_ranks": (
                "; ".join(matched_ranks)
            ),
            "reported_members": (
                log_row["reported_members"]
            ),
            "reconstructed_members": 0,
            "members_with_valid_sequences": 0,
            "member_count_matches_log": False,
            "datasets_contributing_to_group": 0,
            "source_dataset_names": "",
            "representative_found_in_source": False,
            "representative_source_datasets": "",
            "representative_found_in_final_database": (
                representative_found_in_final
            ),
            "final_database_sequence_length": (
                final_database_sequence_length
            ),
            "shortest_discarded_length": pd.NA,
            "shortest_discarded_candidate_ids": "",
            "shortest_discarded_original_ids": "",
            "shortest_discarded_source_datasets": "",
            "length_gain_over_shortest_discarded": pd.NA,
            "reported_representative_length": (
                log_row[
                    "reported_representative_length"
                ]
            ),
            "calculated_representative_length": pd.NA,
            "maximum_group_length": pd.NA,
            "number_of_longest_candidates": 0,
            "longest_candidate_ids": "",
            "longest_candidate_original_ids": "",
            "longest_candidate_source_datasets": "",
            "representative_is_longest": False,
            "length_matches_log": False,
        })

        print(
            f"Group {group_number}: "
            "GROUP_NOT_RECONSTRUCTED"
        )

        continue

    valid_sequence_members = group_members[
        group_members["_calculated_length"]
        .gt(0)
    ].copy()

    # --------------------------------------------------------
    # Group without usable sequences
    # --------------------------------------------------------

    if valid_sequence_members.empty:

        validation_rows.append({
            "group_number": group_number,
            "domain": log_row.get("domain", ""),
            "phylum": log_row.get("phylum", ""),
            "class": log_row.get("class", ""),
            "order": log_row.get("order", ""),
            "family": log_row.get("family", ""),
            "genus": log_row.get("genus", ""),
            "species": log_row.get("species", ""),
            "taxonomic_key": log_row["taxonomic_key"],
            "representative": representative,
            "status": "GROUP_WITHOUT_VALID_SEQUENCES",
            "matched_taxonomic_ranks": (
                "; ".join(matched_ranks)
            ),
            "reported_members": (
                log_row["reported_members"]
            ),
            "reconstructed_members": (
                len(group_members)
            ),
            "members_with_valid_sequences": 0,
            "member_count_matches_log": False,
            "datasets_contributing_to_group": (
                group_members[
                    "source_dataset"
                ].nunique()
            ),
            "source_dataset_names": "; ".join(
                sorted(
                    group_members[
                        "source_dataset"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
            "representative_found_in_source": False,
            "representative_source_datasets": "",
            "representative_found_in_final_database": (
                representative_found_in_final
            ),
            "final_database_sequence_length": (
                final_database_sequence_length
            ),
            "shortest_discarded_length": pd.NA,
            "shortest_discarded_candidate_ids": "",
            "shortest_discarded_original_ids": "",
            "shortest_discarded_source_datasets": "",
            "length_gain_over_shortest_discarded": pd.NA,
            "reported_representative_length": (
                log_row[
                    "reported_representative_length"
                ]
            ),
            "calculated_representative_length": pd.NA,
            "maximum_group_length": pd.NA,
            "number_of_longest_candidates": 0,
            "longest_candidate_ids": "",
            "longest_candidate_original_ids": "",
            "longest_candidate_source_datasets": "",
            "representative_is_longest": False,
            "length_matches_log": False,
        })

        print(
            f"Group {group_number}: "
            "GROUP_WITHOUT_VALID_SEQUENCES"
        )

        continue

    # --------------------------------------------------------
    # Identify maximum sequence length
    # --------------------------------------------------------

    group_max_length = int(
        valid_sequence_members[
            "_calculated_length"
        ].max()
    )

    longest_members = (
        valid_sequence_members[
            valid_sequence_members[
                "_calculated_length"
            ].eq(group_max_length)
        ]
        .copy()
    )

    # Identify candidates that were discarded rather than retained.
    # The representative can be identified either by OTU_XX or
    # by Original_ID, depending on the Collapse Log.
    representative_candidate_mask = (
        valid_sequence_members["_feature_id"]
        .eq(representative)
        |
        valid_sequence_members["_original_id"]
        .eq(representative)
    )

    discarded_members = valid_sequence_members[
        ~representative_candidate_mask
    ].copy()

    if not discarded_members.empty:

        shortest_discarded_length = int(
            discarded_members[
                "_calculated_length"
            ].min()
        )

        shortest_discarded_members = discarded_members[
            discarded_members[
                "_calculated_length"
            ].eq(shortest_discarded_length)
        ].copy()

        length_gain_over_shortest_discarded = (
            group_max_length
            - shortest_discarded_length
        )

    else:

        shortest_discarded_length = pd.NA
        shortest_discarded_members = pd.DataFrame()
        length_gain_over_shortest_discarded = pd.NA

    # The representative may correspond to OTU_XX
    representative_rows = (
        valid_sequence_members[
            valid_sequence_members[
                "_feature_id"
            ].eq(representative)
        ]
        .copy()
    )

    # Additional fallback:
    # search Original_ID if it was the identifier reported by log
    if representative_rows.empty:

        representative_rows = (
            valid_sequence_members[
                valid_sequence_members[
                    "_original_id"
                ].eq(representative)
            ]
            .copy()
        )

    representative_found_in_source = (
        not representative_rows.empty
    )

    if representative_found_in_source:

        calculated_representative_length = int(
            representative_rows[
                "_calculated_length"
            ].max()
        )

    else:

        calculated_representative_length = pd.NA

    reported_length = log_row[
        "reported_representative_length"
    ]

    representative_is_longest = bool(
        representative_found_in_source
        and calculated_representative_length
        == group_max_length
    )

    length_matches_log = bool(
        representative_found_in_source
        and not pd.isna(reported_length)
        and calculated_representative_length
        == int(reported_length)
    )

    if pd.isna(
        log_row["reported_members"]
    ):

        member_count_matches_log = pd.NA

    else:

        member_count_matches_log = (
            len(group_members)
            == int(log_row["reported_members"])
        )

    # --------------------------------------------------------
    # Validation status
    # --------------------------------------------------------

    if not representative_found_in_source:

        status = (
            "REPRESENTATIVE_NOT_FOUND_IN_SOURCE"
        )

    elif not representative_is_longest:

        status = "FAIL_NOT_LONGEST"

    elif not representative_found_in_final:

        status = (
            "LONGEST_BUT_NOT_FOUND_IN_FINAL_DB"
        )

    elif not length_matches_log:

        status = (
            "LONGEST_BUT_LOG_LENGTH_MISMATCH"
        )

    elif member_count_matches_log is False:

        status = (
            "PASS_LONGEST_GROUP_COUNT_MISMATCH"
        )

    elif len(longest_members) > 1:

        status = "PASS_LONGEST_TIE"

    else:

        status = "PASS_UNIQUE_LONGEST"

    representative_source_datasets = (
        "; ".join(
            sorted(
                representative_rows[
                    "source_dataset"
                ]
                .dropna()
                .astype(str)
                .unique()
            )
        )
        if representative_found_in_source
        else ""
    )

    # --------------------------------------------------------
    # Group-level validation record
    # --------------------------------------------------------

    validation_rows.append({
        "group_number": group_number,
        "domain": log_row.get("domain", ""),
        "phylum": log_row.get("phylum", ""),
        "class": log_row.get("class", ""),
        "order": log_row.get("order", ""),
        "family": log_row.get("family", ""),
        "genus": log_row.get("genus", ""),
        "species": log_row.get("species", ""),
        "taxonomic_key": log_row["taxonomic_key"],
        "representative": representative,
        "status": status,
        "matched_taxonomic_ranks": (
            "; ".join(matched_ranks)
        ),
        "reported_members": (
            log_row["reported_members"]
        ),
        "reconstructed_members": (
            len(group_members)
        ),
        "members_with_valid_sequences": (
            len(valid_sequence_members)
        ),
        "member_count_matches_log": (
            member_count_matches_log
        ),
        "datasets_contributing_to_group": (
            group_members[
                "source_dataset"
            ].nunique()
        ),
        "source_dataset_names": "; ".join(
            sorted(
                group_members[
                    "source_dataset"
                ]
                .dropna()
                .astype(str)
                .unique()
            )
        ),
        "representative_found_in_source": (
            representative_found_in_source
        ),
        "representative_source_datasets": (
            representative_source_datasets
        ),
        "representative_found_in_final_database": (
            representative_found_in_final
        ),
        "final_database_sequence_length": (
            final_database_sequence_length
        ),
        "shortest_discarded_length": (
            shortest_discarded_length
        ),
        "shortest_discarded_candidate_ids": (
            "; ".join(
                shortest_discarded_members[
                    "_feature_id"
                ]
                .astype(str)
                .drop_duplicates()
                .tolist()
            )
            if not shortest_discarded_members.empty
            else ""
        ),
        "shortest_discarded_original_ids": (
            "; ".join(
                shortest_discarded_members[
                    "_original_id"
                ]
                .astype(str)
                .drop_duplicates()
                .tolist()
            )
            if not shortest_discarded_members.empty
            else ""
        ),
        "shortest_discarded_source_datasets": (
            "; ".join(
                sorted(
                    shortest_discarded_members[
                        "source_dataset"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            )
            if not shortest_discarded_members.empty
            else ""
        ),
        "length_gain_over_shortest_discarded": (
            length_gain_over_shortest_discarded
        ),
        "reported_representative_length": (
            reported_length
        ),
        "calculated_representative_length": (
            calculated_representative_length
        ),
        "maximum_group_length": (
            group_max_length
        ),
        "number_of_longest_candidates": (
            len(longest_members)
        ),
        "longest_candidate_ids": "; ".join(
            longest_members[
                "_feature_id"
            ]
            .astype(str)
            .drop_duplicates()
            .tolist()
        ),
        "longest_candidate_original_ids": "; ".join(
            longest_members[
                "_original_id"
            ]
            .astype(str)
            .drop_duplicates()
            .tolist()
        ),
        "longest_candidate_source_datasets": (
            "; ".join(
                sorted(
                    longest_members[
                        "source_dataset"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            )
        ),
        "representative_is_longest": (
            representative_is_longest
        ),
        "length_matches_log": (
            length_matches_log
        ),
        "final_database_length_matches_representative": (
            bool(
                representative_found_in_source
                and representative_found_in_final
                and not pd.isna(
                    final_database_sequence_length
                )
                and final_database_sequence_length
                == calculated_representative_length
            )
        ),
    })

    # --------------------------------------------------------
    # Shortest discarded candidate table
    # --------------------------------------------------------

    if not shortest_discarded_members.empty:

        shortest_detail = shortest_discarded_members.copy()

        shortest_detail.insert(
            0,
            "group_number",
            group_number,
        )

        shortest_detail.insert(
            1,
            "taxonomic_key",
            log_row["taxonomic_key"],
        )

        shortest_detail.insert(
            2,
            "retained_representative",
            representative,
        )

        shortest_detail.insert(
            3,
            "retained_sequence_length",
            calculated_representative_length,
        )

        shortest_detail.insert(
            4,
            "final_database_sequence_length",
            final_database_sequence_length,
        )

        shortest_detail.insert(
            5,
            "shortest_discarded_length",
            shortest_discarded_length,
        )

        shortest_detail.insert(
            6,
            "length_gain_bp",
            length_gain_over_shortest_discarded,
        )

        shortest_discarded_tables.append(
            shortest_detail
        )

    # --------------------------------------------------------
    # Detailed members table
    # --------------------------------------------------------

    group_detail = group_members.copy()

    group_detail.insert(
        0,
        "group_number",
        group_number,
    )

    group_detail.insert(
        1,
        "taxonomic_key",
        log_row["taxonomic_key"],
    )

    group_detail.insert(
        2,
        "reported_representative",
        representative,
    )

    representative_match_mask = (
        group_detail["_feature_id"]
        .eq(representative)
        |
        group_detail["_original_id"]
        .eq(representative)
    )

    group_detail.insert(
        3,
        "is_reported_representative",
        representative_match_mask,
    )

    group_detail.insert(
        4,
        "is_longest_candidate",
        group_detail["_calculated_length"]
        .eq(group_max_length),
    )

    group_detail.insert(
        5,
        "maximum_group_length",
        group_max_length,
    )

    group_detail[
        "length_rank_within_group"
    ] = (
        group_detail[
            "_calculated_length"
        ]
        .rank(
            method="min",
            ascending=False,
        )
        .astype("Int64")
    )

    group_detail[
        "length_difference_from_maximum"
    ] = (
        group_max_length
        - group_detail[
            "_calculated_length"
        ]
    )

    group_detail = group_detail.sort_values(
        by=[
            "_calculated_length",
            "source_dataset",
            "_feature_id",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )

    member_tables.append(
        group_detail
    )

    # --------------------------------------------------------
    # Counts by dataset and group
    # --------------------------------------------------------

    dataset_counts = (
        group_members
        .groupby(
            [
                "source_dataset",
                "source_file",
            ],
            dropna=False,
        )
        .agg(
            candidate_rows=(
                "_dataset_feature_key",
                "size",
            ),
            unique_candidate_features=(
                "_dataset_feature_key",
                "nunique",
            ),
            unique_feature_ids=(
                "_feature_id",
                "nunique",
            ),
            unique_original_ids=(
                "_original_id",
                "nunique",
            ),
            valid_sequences=(
                "_calculated_length",
                lambda values: int(
                    values.gt(0).sum()
                ),
            ),
            minimum_sequence_length=(
                "_calculated_length",
                "min",
            ),
            maximum_sequence_length=(
                "_calculated_length",
                "max",
            ),
            mean_sequence_length=(
                "_calculated_length",
                "mean",
            ),
        )
        .reset_index()
    )

    dataset_counts.insert(
        0,
        "group_number",
        group_number,
    )

    dataset_counts.insert(
        1,
        "taxonomic_key",
        log_row["taxonomic_key"],
    )

    dataset_counts.insert(
        2,
        "reported_representative",
        representative,
    )

    representative_datasets = set(
        representative_rows[
            "source_dataset"
        ].astype(str)
    )

    longest_datasets = set(
        longest_members[
            "source_dataset"
        ].astype(str)
    )

    dataset_counts[
        "contains_representative"
    ] = (
        dataset_counts[
            "source_dataset"
        ]
        .astype(str)
        .isin(representative_datasets)
    )

    dataset_counts[
        "contains_longest_candidate"
    ] = (
        dataset_counts[
            "source_dataset"
        ]
        .astype(str)
        .isin(longest_datasets)
    )

    dataset_counts[
        "representative_is_longest"
    ] = representative_is_longest

    dataset_count_tables.append(
        dataset_counts
    )

    print(
        f"Group {group_number}: {status}"
    )


# ============================================================
# 13. CREATE OUTPUT TABLES
# ============================================================

validation_table = pd.DataFrame(
    validation_rows
)

if member_tables:

    members_table = pd.concat(
        member_tables,
        ignore_index=True,
        sort=False,
    )

else:

    members_table = pd.DataFrame()

if dataset_count_tables:

    dataset_counts_table = pd.concat(
        dataset_count_tables,
        ignore_index=True,
        sort=False,
    )

else:

    dataset_counts_table = pd.DataFrame()

if shortest_discarded_tables:

    shortest_discarded_table = pd.concat(
        shortest_discarded_tables,
        ignore_index=True,
        sort=False,
    )

else:

    shortest_discarded_table = pd.DataFrame()


# ============================================================
# 14. OVERALL DATASET CONTRIBUTION
# ============================================================

if not members_table.empty:

    overall_dataset_counts = (
        members_table
        .groupby(
            [
                "source_dataset",
                "source_file",
            ],
            dropna=False,
        )
        .agg(
            collapse_groups_contributed=(
                "group_number",
                "nunique",
            ),
            total_candidate_rows=(
                "_dataset_feature_key",
                "size",
            ),
            unique_candidate_features=(
                "_dataset_feature_key",
                "nunique",
            ),
            unique_feature_ids=(
                "_feature_id",
                "nunique",
            ),
            unique_original_ids=(
                "_original_id",
                "nunique",
            ),
            times_reported_as_representative=(
                "is_reported_representative",
                "sum",
            ),
            times_containing_longest_candidate=(
                "is_longest_candidate",
                "sum",
            ),
            minimum_sequence_length=(
                "_calculated_length",
                "min",
            ),
            maximum_sequence_length=(
                "_calculated_length",
                "max",
            ),
            mean_sequence_length=(
                "_calculated_length",
                "mean",
            ),
        )
        .reset_index()
    )

else:

    overall_dataset_counts = pd.DataFrame()


# ============================================================
# 15. GENERAL SUMMARY
# ============================================================

if validation_table.empty:
    raise ValueError(
        "No validation records were generated."
    )

passed_mask = (
    validation_table[
        "representative_is_longest"
    ]
    .fillna(False)
    .astype(bool)
)

failed_mask = ~passed_mask

summary_table = pd.DataFrame([{
    "target_rank": TARGET_RANK,
    "target_name": TARGET_NAME,
    "subtarget_rank": (
        SUBTARGET_RANK
        if SUBTARGET_RANK is not None
        else ""
    ),
    "subtarget_name": (
        SUBTARGET_NAME
        if SUBTARGET_NAME is not None
        else ""
    ),
    "original_datasets_loaded": (
        precollapse_database[
            "source_dataset"
        ].nunique()
    ),
    "original_rows_loaded": (
        len(precollapse_database)
    ),
    "collapse_groups_evaluated": (
        len(validation_table)
    ),
    "groups_passed": int(
        passed_mask.sum()
    ),
    "groups_failed": int(
        failed_mask.sum()
    ),
    "unique_longest_representatives": int(
        validation_table[
            "status"
        ]
        .eq("PASS_UNIQUE_LONGEST")
        .sum()
    ),
    "longest_representatives_with_ties": int(
        validation_table[
            "status"
        ]
        .eq("PASS_LONGEST_TIE")
        .sum()
    ),
    "group_count_mismatches": int(
        validation_table[
            "status"
        ]
        .eq(
            "PASS_LONGEST_GROUP_COUNT_MISMATCH"
        )
        .sum()
    ),
    "representatives_not_longest": int(
        validation_table[
            "status"
        ]
        .eq("FAIL_NOT_LONGEST")
        .sum()
    ),
    "representatives_missing_from_source": int(
        validation_table[
            "representative_found_in_source"
        ]
        .fillna(False)
        .eq(False)
        .sum()
    ),
    "representatives_missing_from_final_database": int(
        validation_table[
            "representative_found_in_final_database"
        ]
        .fillna(False)
        .eq(False)
        .sum()
    ),
    "groups_not_reconstructed": int(
        validation_table[
            "status"
        ]
        .eq("GROUP_NOT_RECONSTRUCTED")
        .sum()
    ),
    "groups_without_valid_sequences": int(
        validation_table[
            "status"
        ]
        .eq("GROUP_WITHOUT_VALID_SEQUENCES")
        .sum()
    ),
    "all_representatives_are_longest": bool(
        passed_mask.all()
    ),
    "final_database_lengths_match_representatives": bool(
        validation_table[
            "final_database_length_matches_representative"
        ]
        .fillna(False)
        .all()
    ),
    "minimum_shortest_discarded_length": (
        pd.to_numeric(
            validation_table[
                "shortest_discarded_length"
            ],
            errors="coerce",
        ).min()
    ),
    "maximum_length_gain_bp": (
        pd.to_numeric(
            validation_table[
                "length_gain_over_shortest_discarded"
            ],
            errors="coerce",
        ).max()
    ),
}])


# ============================================================
# 16. OUTPUT FILE NAMES
# ============================================================

target_label = safe_name(
    TARGET_NAME
)

run_label = safe_name(
    RUN_EPITHET
)

if SUBTARGET_NAME is not None:
    target_label += (
        "_"
        + safe_name(SUBTARGET_NAME)
    )

summary_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}_summary.csv"
    )
)

validation_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}_groups.csv"
    )
)

members_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}_members.csv"
    )
)

shortest_discarded_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}_shortest_discarded.csv"
    )
)

dataset_counts_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}"
        "_counts_by_dataset_and_group.csv"
    )
)

overall_dataset_counts_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}"
        "_overall_dataset_counts.csv"
    )
)

source_dataset_summary_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}"
        "_input_dataset_summary.csv"
    )
)

source_dataset_issues_path = (
    OUTPUT_FOLDER
    / (
        "LongestRepresentativeValidation_"
        f"{run_label}_{target_label}"
        "_input_dataset_issues.csv"
    )
)


# ============================================================
# 17. SAVE OUTPUTS
# ============================================================

summary_table.to_csv(
    summary_path,
    index=False,
)

validation_table.to_csv(
    validation_path,
    index=False,
)

members_table.to_csv(
    members_path,
    index=False,
)

shortest_discarded_table.to_csv(
    shortest_discarded_path,
    index=False,
)

dataset_counts_table.to_csv(
    dataset_counts_path,
    index=False,
)

overall_dataset_counts.to_csv(
    overall_dataset_counts_path,
    index=False,
)

source_dataset_summary.to_csv(
    source_dataset_summary_path,
    index=False,
)

if not source_dataset_issues.empty:

    source_dataset_issues.to_csv(
        source_dataset_issues_path,
        index=False,
    )


# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print(
    "\n===================================================="
)

print(
    "Longest representative validation completed."
)

print(
    "===================================================="
)

print(
    "\nOriginal datasets loaded: "
    f"{precollapse_database['source_dataset'].nunique():,}"
)

print(
    "Original rows loaded: "
    f"{len(precollapse_database):,}"
)

print(
    "Collapse groups evaluated: "
    f"{len(validation_table):,}"
)

print(
    "Groups with a longest representative: "
    f"{int(passed_mask.sum()):,}"
)

print(
    "Groups failing the longest-sequence criterion: "
    f"{int(failed_mask.sum()):,}"
)

print(
    "\nSummary:"
)

display(
    summary_table
)

print(
    "\nValidation by collapse group:"
)

display(
    validation_table
)

if not shortest_discarded_table.empty:

    print(
        "\nShortest discarded candidate sequence "
        "for each collapse group:"
    )

    display(
        shortest_discarded_table[
            [
                "group_number",
                "retained_representative",
                "retained_sequence_length",
                "final_database_sequence_length",
                "_feature_id",
                "_original_id",
                "source_dataset",
                "shortest_discarded_length",
                "length_gain_bp",
            ]
        ]
    )

if not dataset_counts_table.empty:

    print(
        "\nCandidate counts by original dataset "
        "and collapse group:"
    )

    display(
        dataset_counts_table
    )

if not overall_dataset_counts.empty:

    print(
        "\nOverall contribution of each "
        "original dataset:"
    )

    display(
        overall_dataset_counts
    )

failed_groups = validation_table[
    ~validation_table[
        "representative_is_longest"
    ]
    .fillna(False)
    .astype(bool)
]

if not failed_groups.empty:

    print(
        "\nWARNING — groups that failed validation:"
    )

    display(
        failed_groups
    )

else:

    print(
        "\nPASS: every evaluated representative "
        "corresponds to a longest sequence in its "
        "reconstructed group."
    )

if not source_dataset_issues.empty:

    print(
        "\nWARNING — original datasets that could "
        "not be loaded:"
    )

    display(
        source_dataset_issues
    )


# ============================================================
# 19. DISPLAY TARGET EXAMPLE RECORDS
# ============================================================

target_example = subset_taxonomic_group(
    precollapse_database,
    TARGET_RANK,
    TARGET_NAME,
)

if (
    SUBTARGET_RANK is not None
    and SUBTARGET_NAME is not None
):
    target_example = subset_taxonomic_group(
        target_example,
        SUBTARGET_RANK,
        SUBTARGET_NAME,
    )

example_columns = [
    "source_dataset",
    "_feature_id",
    "_original_id",
    "_sintax_taxonomy",
    "domain",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
    "_calculated_length",
]

example_columns = [
    column
    for column in example_columns
    if column in target_example.columns
]

if not target_example.empty:

    print(
        "\nExample records for the selected taxonomic target:"
    )

    display(
        target_example[example_columns].head(20)
    )


# ============================================================
# 20. PRINT OUTPUT PATHS
# ============================================================

print(
    f"\nSummary saved to:\n{summary_path}"
)

print(
    f"\nGroup validation saved to:\n"
    f"{validation_path}"
)

print(
    f"\nAll candidate members saved to:\n"
    f"{members_path}"
)

print(
    f"\nShortest discarded candidates saved to:\n"
    f"{shortest_discarded_path}"
)

print(
    f"\nCounts by dataset and group saved to:\n"
    f"{dataset_counts_path}"
)

print(
    f"\nOverall dataset counts saved to:\n"
    f"{overall_dataset_counts_path}"
)

print(
    f"\nInput dataset summary saved to:\n"
    f"{source_dataset_summary_path}"
)

if not source_dataset_issues.empty:

    print(
        f"\nInput dataset issues saved to:\n"
        f"{source_dataset_issues_path}"
    )

Project directory: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC
Marker mode: ITS
Subset mode: all_eukaryotes
Run epithet: genus_all_eukaryotes_p1
Pre-collapse folder: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\concatenated_tables
Final Database: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\Final_Database_genus_all_eukaryotes_p1.csv
Collapse Log: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\Collapse_Log_genus_all_eukaryotes_p1.txt
Validation output folder: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation

Reading original pre-collapse datasets...
Loaded: ATLASMXA_concatenated.csv | rows: 54,756 | target rows: 91
Loaded: ATLASMXBC_concatenated.csv | rows: 82,000 | target rows: 105

Reading Final Database...

Reading Collap

,source_dataset,source_file,rows_loaded,unique_feature_ids,empty_feature_ids,unique_dataset_feature_keys,duplicate_dataset_feature_keys,sequences_with_length_gt_zero,empty_sequences,minimum_sequence_length,maximum_sequence_length,target_rows,subtarget_rows,status
0,ATLASMXA,ATLASMXA_concatenated.csv,54756,54756,0,54756,0,54756,0,112,1751,91,84,LOADED
1,ATLASMXBC,ATLASMXBC_concatenated.csv,82000,82000,0,82000,0,82000,0,197,550,105,83,LOADED



Total original rows loaded: 136,756
Total original datasets loaded: 2

Selected collapse groups: 1
Group 119: PASS_LONGEST_GROUP_COUNT_MISMATCH

Longest representative validation completed.

Original datasets loaded: 2
Original rows loaded: 136,756
Collapse groups evaluated: 1
Groups with a longest representative: 1
Groups failing the longest-sequence criterion: 0

Summary:


,target_rank,target_name,subtarget_rank,subtarget_name,original_datasets_loaded,original_rows_loaded,collapse_groups_evaluated,groups_passed,groups_failed,unique_longest_representatives,...,group_count_mismatches,representatives_not_longest,representatives_missing_from_source,representatives_missing_from_final_database,groups_not_reconstructed,groups_without_valid_sequences,all_representatives_are_longest,final_database_lengths_match_representatives,minimum_shortest_discarded_length,maximum_length_gain_bp
0,family,Amanitaceae,genus,Amanita,2,136756,1,1,0,0,...,1,0,0,0,0,0,True,True,294,358



Validation by collapse group:


...


Overall contribution of each original dataset:


,source_dataset,source_file,collapse_groups_contributed,total_candidate_rows,unique_candidate_features,unique_feature_ids,unique_original_ids,times_reported_as_representative,times_containing_longest_candidate,minimum_sequence_length,maximum_sequence_length,mean_sequence_length
0,ATLASMXA,ATLASMXA_concatenated.csv,1,84,84,84,84,1,1,471,652,558.702381
1,ATLASMXBC,ATLASMXBC_concatenated.csv,1,83,83,83,83,0,0,294,398,334.084337



PASS: every evaluated representative corresponds to a longest sequence in its reconstructed group.

Example records for the selected taxonomic target:


,source_dataset,_feature_id,_original_id,_sintax_taxonomy,domain,phylum,class,order,family,genus,species,_calculated_length
529,ATLASMXA,ATLASMXA_ID25570,Amanita_griseoumbonata_10_bbcc00c551a19b829ec1...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,516
738,ATLASMXA,ATLASMXA_ID25625,Amanita_sp_8_adc60886a933dfe6a0866a77175e8db8e...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,592
781,ATLASMXA,ATLASMXA_ID25590,Amanita_prudens_4_94d4eba29eaa1319c47f3a5827d5...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,497
922,ATLASMXA,ATLASMXA_ID25632,Amanita_sp_15_fdbabf4eed97af45f7f4e80b1db25923...,"d:Fungi(0.92),p:Basidiomycota(0.92),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,arenicola,518
1021,ATLASMXA,ATLASMXA_ID25588,Amanita_prudens_2_38974d254c81a973c24d8e300e81...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,518
1035,ATLASMXA,ATLASMXA_ID25635,Amanita_submembranacea_3_a5ac80e033afcded760f3...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,493
1111,ATLASMXA,ATLASMXA_ID25607,Amanita_rubescens_8_bc4f959f9f3bfe700b4672c061...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,634
1229,ATLASMXA,ATLASMXA_ID25633,Amanita_submembranacea_1_7d4037e527663f1fb9303...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,491
1389,ATLASMXA,ATLASMXA_ID25608,Amanita_rubescens_9_c36d130e31b3e8f3c4fd1414d0...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,novinupta,641
1534,ATLASMXA,ATLASMXA_ID25622,Amanita_sp_20_8cfa96882c5bcafbe08375fcc9523f06...,"d:Fungi(1.00),p:Basidiomycota(1.00),c:Agaricom...",Fungi,Basidiomycota,Agaricomycetes,Agaricales,Amanitaceae,Amanita,,501



Summary saved to:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\LongestRepresentativeValidation_genus_all_eukaryotes_p1_Amanitaceae_Amanita_summary.csv

Group validation saved to:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\LongestRepresentativeValidation_genus_all_eukaryotes_p1_Amanitaceae_Amanita_groups.csv

All candidate members saved to:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\LongestRepresentativeValidation_genus_all_eukaryotes_p1_Amanitaceae_Amanita_members.csv

Shortest discarded candidates saved to:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\LongestRepresentativeValidation_genus_all_eukaryotes_p1_Amanitaceae_Amanita_shortest_discarded.csv

Counts by dataset and group saved to:
C:\Users\berna\Desktop

# MetaDiv Builder — Standardized Biodiversity-Unit Identifier Validation SPPN

This notebook automatically resolves the MetaDiv Builder project directory and selects the Final Database from the configured marker, subset mode, collapse strategy, and confidence threshold.

Validation files are written to `output/<MODE>/FINAL_DB/validation/`.

In [6]:
from pathlib import Path
import os
import re
import pandas as pd
from IPython.display import display


# ============================================================
# MetaDiv Builder
# Standardized Biodiversity-Unit Identifier Validation v1.3
#
# Purpose
# -------
# Validate that standardized biodiversity-unit identifiers:
#
# 1. are unique;
# 2. use the same prefix-generation rule as MetaDiv Builder;
# 3. treat unassigned units as Unknown;
# 4. contain a valid sequential numerical suffix;
# 5. use consecutive suffixes within each Builder prefix;
# 6. follow decreasing cumulative abundance;
# 7. correctly allow tied abundances in any order within
#    their corresponding numerical interval.
#
# Required input
# --------------
# One MetaDiv Builder Final Database CSV.
#
# Main expected columns
# ---------------------
# - standardized identifier, normally SPPN
# - Total_Abundance
# - taxonomic columns:
#   domain, phylum, class, order, family, genus, species
# ============================================================


# ============================================================
# 1. USER SETTINGS AND PROJECT PATHS
# ============================================================

MODE = "ITS"  # "ITS", "16S", or "CO1"

# Valid subset modes depend on MODE:
# ITS: "only_fungi" or "all_eukaryotes"
# 16S: "only_bacteria" or "all_prokaryotes"
# CO1: "only_metazoa" or "all_eukaryotes"
SUBSET_MODE = "all_eukaryotes"

COLLAPSE_STRATEGY = "genus"  # "species_only", "genus", or "all"
P_VALUE_THRESHOLD = 1.00

# Usually the standardized identifier column is named SPPN.
# Leave as None to detect it automatically.
STANDARDIZED_ID_COLUMN = None

# Usually cumulative abundance is stored in Total_Abundance.
# Leave as None to detect it automatically.
TOTAL_ABUNDANCE_COLUMN = None

# Number of digits expected in the numerical suffix.
EXPECTED_SUFFIX_DIGITS = 4

# Optional taxonomic restriction.
# Use None / None to validate the complete Final Database.
FILTER_RANK = None
FILTER_NAME = None


def resolve_project_directory() -> Path:
    """
    Resolve the MetaDiv Builder project directory.

    The METADIV_PROJECT_DIR environment variable has priority.
    Otherwise, the current working directory and its parent
    directories are searched for a project containing an
    ``output`` directory.
    """
    configured_directory = os.environ.get("METADIV_PROJECT_DIR")

    if configured_directory:
        project_directory = Path(
            configured_directory
        ).expanduser().resolve()

        if not project_directory.exists():
            raise FileNotFoundError(
                "METADIV_PROJECT_DIR does not exist:\n"
                f"{project_directory}"
            )

        return project_directory

    current_directory = Path.cwd().resolve()

    for candidate in (
        current_directory,
        *current_directory.parents,
    ):
        if (candidate / "output").is_dir():
            return candidate

    return current_directory


def threshold_token(value) -> str:
    """
    Convert a numeric threshold to the MetaDiv file-name token.

    Examples
    --------
    1.00 -> p1
    0.80 -> p08
    0.50 -> p05
    0.00 -> p0
    """
    numeric_value = float(value)

    if not 0.0 <= numeric_value <= 1.0:
        raise ValueError(
            "P_VALUE_THRESHOLD must be between 0.00 and 1.00."
        )

    formatted = f"{numeric_value:.2f}".rstrip("0").rstrip(".")

    if formatted.startswith("0."):
        formatted = formatted.replace("0.", "0", 1)

    return f"p{formatted}"


def build_run_epithet() -> str:
    """
    Build the complete MetaDiv run epithet used in output files.
    """
    return "_".join(
        [
            COLLAPSE_STRATEGY,
            SUBSET_MODE,
            threshold_token(P_VALUE_THRESHOLD),
        ]
    )


PROJECT_DIR = resolve_project_directory()
FINAL_DB_FOLDER = PROJECT_DIR / "output" / MODE / "FINAL_DB"
OUTPUT_FOLDER = FINAL_DB_FOLDER / "validation"

RUN_EPITHET = build_run_epithet()

FINAL_DATABASE_FILE = (
    FINAL_DB_FOLDER
    / f"Final_Database_{RUN_EPITHET}.csv"
)

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. CONSTANTS
# ============================================================

TAXONOMIC_RANKS = [
    "domain",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
]

STANDARDIZED_ID_CANDIDATES = [
    "SPPN",
    "sppn",
    "standardized_id",
    "standardized_identifier",
    "standardized_biodiversity_unit",
    "biodiversity_unit_id",
    "feature_id",
]

TOTAL_ABUNDANCE_CANDIDATES = [
    "Total_Abundance",
    "total_abundance",
    "Total abundance",
    "total abundance",
    "cumulative_abundance",
    "Cumulative_Abundance",
]

MISSING_TAXON_TERMS = {
    "",
    "na",
    "nan",
    "none",
    "null",
    "unassigned",
    "unclassified",
    "unidentified",
    "unknown",
    "uncultured",
    "incertae_sedis",
    "incertae sedis",
}


# ============================================================
# 3. VALIDATE USER SETTINGS
# ============================================================

VALID_SUBSET_MODES = {
    "ITS": {"only_fungi", "all_eukaryotes"},
    "16S": {"only_bacteria", "all_prokaryotes"},
    "CO1": {"only_metazoa", "all_eukaryotes"},
}

VALID_COLLAPSE_STRATEGIES = {
    "species_only",
    "genus",
    "all",
}

MODE = MODE.strip().upper()
SUBSET_MODE = SUBSET_MODE.strip().casefold()
COLLAPSE_STRATEGY = COLLAPSE_STRATEGY.strip().casefold()

if MODE not in VALID_SUBSET_MODES:
    raise ValueError(
        "MODE must be one of: ITS, 16S, or CO1."
    )

if SUBSET_MODE not in VALID_SUBSET_MODES[MODE]:
    raise ValueError(
        f"SUBSET_MODE '{SUBSET_MODE}' is not valid for MODE "
        f"'{MODE}'. Accepted values: "
        f"{sorted(VALID_SUBSET_MODES[MODE])}"
    )

if COLLAPSE_STRATEGY not in VALID_COLLAPSE_STRATEGIES:
    raise ValueError(
        "COLLAPSE_STRATEGY must be one of: "
        f"{sorted(VALID_COLLAPSE_STRATEGIES)}"
    )

if FILTER_RANK is not None:
    FILTER_RANK = FILTER_RANK.strip().casefold()


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def detect_column(table, requested_column, candidates, label):
    if requested_column is not None:
        if requested_column not in table.columns:
            raise KeyError(
                f"{label} column '{requested_column}' was not found."
            )
        return requested_column

    exact = {str(column): column for column in table.columns}
    folded = {
        str(column).strip().casefold(): column
        for column in table.columns
    }

    for candidate in candidates:
        if candidate in exact:
            return exact[candidate]

        candidate_folded = candidate.strip().casefold()
        if candidate_folded in folded:
            return folded[candidate_folded]

    raise KeyError(
        f"Could not detect the {label} column. "
        f"Available columns: {list(table.columns)}"
    )


def standardize_taxonomic_columns(table):
    table = table.copy()
    folded = {
        str(column).strip().casefold(): column
        for column in table.columns
    }

    rename_map = {}

    for rank in TAXONOMIC_RANKS:
        if rank in folded:
            rename_map[folded[rank]] = rank

    table = table.rename(columns=rename_map)

    missing_ranks = [
        rank for rank in TAXONOMIC_RANKS
        if rank not in table.columns
    ]

    if missing_ranks:
        raise KeyError(
            "The following taxonomic columns were not found: "
            + ", ".join(missing_ranks)
        )

    return table


def clean_taxon(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    # Remove common rank prefixes when present.
    text = re.sub(
        r"^[dkpcofgs]__?",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    if text.casefold() in MISSING_TAXON_TERMS:
        return None

    return text


def identifier_safe_taxon(value):
    """
    Convert a taxonomic name exactly as MetaDiv Builder does
    inside assign_sppn_unique().

    Builder rule:
        re.sub(r"[^\\w]+", "_", base_taxon)

    Important:
    - Unicode word characters are preserved;
    - existing underscores are preserved;
    - leading or trailing underscores are not stripped;
    - repeated punctuation is replaced by one underscore.
    """
    value = clean_taxon(value)

    if value is None:
        return "Unknown"

    return re.sub(
        r"[^\w]+",
        "_",
        str(value),
    )


def deepest_taxonomic_assignment(row):
    for rank in reversed(TAXONOMIC_RANKS):
        value = clean_taxon(row.get(rank))

        if value is not None:
            return rank, value

    return None, None


def parse_standardized_identifier(identifier):
    if pd.isna(identifier):
        return None, None, None

    text = str(identifier).strip()

    match = re.match(r"^(.*)_([0-9]+)$", text)

    if match is None:
        return text, None, None

    prefix = match.group(1)
    suffix_text = match.group(2)

    return prefix, int(suffix_text), len(suffix_text)


def normalized_comparison_text(value):
    if value is None or pd.isna(value):
        return None

    return str(value).strip().casefold()


def bool_all(series):
    if len(series) == 0:
        return True

    return bool(series.fillna(False).all())


# ============================================================
# 5. LOAD FINAL DATABASE
# ============================================================

print(f"Project directory: {PROJECT_DIR}")
print(f"Marker mode: {MODE}")
print(f"Subset mode: {SUBSET_MODE}")
print(f"Run epithet: {RUN_EPITHET}")
print(f"Final Database: {FINAL_DATABASE_FILE}")
print(f"Validation output folder: {OUTPUT_FOLDER}")

if not FINAL_DATABASE_FILE.exists():
    available_databases = sorted(
        path.name
        for path in FINAL_DB_FOLDER.glob("Final_Database_*.csv")
    ) if FINAL_DB_FOLDER.exists() else []

    message = (
        "Final Database not found:\n"
        f"{FINAL_DATABASE_FILE}"
    )

    if available_databases:
        message += (
            "\n\nAvailable Final Database files:\n- "
            + "\n- ".join(available_databases)
        )

    raise FileNotFoundError(message)

final_database = pd.read_csv(
    FINAL_DATABASE_FILE,
    low_memory=False,
)

final_database = standardize_taxonomic_columns(
    final_database
)

standardized_id_column = detect_column(
    final_database,
    STANDARDIZED_ID_COLUMN,
    STANDARDIZED_ID_CANDIDATES,
    "standardized identifier",
)

total_abundance_column = detect_column(
    final_database,
    TOTAL_ABUNDANCE_COLUMN,
    TOTAL_ABUNDANCE_CANDIDATES,
    "total abundance",
)

final_database[total_abundance_column] = pd.to_numeric(
    final_database[total_abundance_column],
    errors="coerce",
)


# ============================================================
# 6. OPTIONAL TAXONOMIC FILTER
# ============================================================

validation_database = final_database.copy()

if FILTER_RANK is not None or FILTER_NAME is not None:
    if FILTER_RANK is None or FILTER_NAME is None:
        raise ValueError(
            "FILTER_RANK and FILTER_NAME must either both be "
            "defined or both be None."
        )

    filter_rank = FILTER_RANK.strip().casefold()

    if filter_rank not in TAXONOMIC_RANKS:
        raise ValueError(
            f"FILTER_RANK must be one of: {TAXONOMIC_RANKS}"
        )

    validation_database = validation_database[
        validation_database[filter_rank]
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq(str(FILTER_NAME).strip().casefold())
    ].copy()


# ============================================================
# 7. ROW-LEVEL IDENTIFIER RECONSTRUCTION
# ============================================================

validation_rows = []

for source_index, row in validation_database.iterrows():
    standardized_id = row.get(standardized_id_column)
    abundance = row.get(total_abundance_column)

    deepest_rank, deepest_taxon = deepest_taxonomic_assignment(
        row
    )

    if deepest_taxon is None:
        deepest_rank = "unassigned"
        deepest_taxon = "Unknown"

    expected_prefix = identifier_safe_taxon(
        deepest_taxon
    )

    (
        observed_prefix,
        observed_suffix,
        suffix_digits,
    ) = parse_standardized_identifier(
        standardized_id
    )

    prefix_matches_deepest_taxon = (
        normalized_comparison_text(observed_prefix)
        ==
        normalized_comparison_text(expected_prefix)
    )

    valid_suffix = observed_suffix is not None

    suffix_digit_count_valid = (
        suffix_digits == EXPECTED_SUFFIX_DIGITS
        if suffix_digits is not None
        else False
    )

    validation_rows.append(
        {
            "source_row_index": source_index,
            "standardized_identifier": standardized_id,
            "deepest_accepted_rank": deepest_rank,
            "deepest_accepted_taxon": deepest_taxon,
            "expected_identifier_prefix": expected_prefix,
            "observed_identifier_prefix": observed_prefix,
            "observed_numerical_suffix": observed_suffix,
            "observed_suffix_digits": suffix_digits,
            "total_abundance": abundance,
            "prefix_matches_deepest_taxon":
                prefix_matches_deepest_taxon,
            "valid_numerical_suffix":
                valid_suffix,
            "suffix_digit_count_valid":
                suffix_digit_count_valid,
        }
    )

row_validation = pd.DataFrame(validation_rows)

row_validation[
    "identifier_is_globally_unique"
] = ~row_validation[
    "standardized_identifier"
].duplicated(keep=False)

# MetaDiv Builder increments SPPN counters only by the
# sanitized taxonomic prefix. The taxonomic rank is not part
# of the counter key. Therefore, identical labels occurring at
# different ranks share the same numerical sequence.
row_validation["taxon_group_key"] = (
    row_validation["expected_identifier_prefix"]
    .fillna("Unknown")
    .astype(str)
)


# ============================================================
# 8. GROUP-LEVEL SEQUENTIAL AND ABUNDANCE VALIDATION
# ============================================================

group_results = []

row_validation[
    "suffix_within_abundance_tie_interval"
] = False

row_validation[
    "group_suffixes_are_consecutive"
] = False

row_validation[
    "group_abundance_order_is_valid"
] = False

for group_key, group in row_validation.groupby(
    "taxon_group_key",
    dropna=False,
    sort=False,
):
    group = group.copy()

    group_size = len(group)

    valid_suffix_values = (
        group["observed_numerical_suffix"]
        .dropna()
        .astype(int)
        .tolist()
    )

    expected_suffixes = list(
        range(1, group_size + 1)
    )

    observed_unique_suffixes = sorted(
        set(valid_suffix_values)
    )

    suffixes_are_consecutive = (
        len(valid_suffix_values) == group_size
        and len(observed_unique_suffixes) == group_size
        and observed_unique_suffixes == expected_suffixes
    )

    ordered_by_suffix = group.sort_values(
        "observed_numerical_suffix",
        kind="stable",
        na_position="last",
    )

    ordered_abundances = ordered_by_suffix[
        "total_abundance"
    ]

    abundance_values_complete = bool(
        ordered_abundances.notna().all()
    )

    if abundance_values_complete:
        abundance_order_is_valid = bool(
            ordered_abundances
            .diff()
            .dropna()
            .le(0)
            .all()
        )
    else:
        abundance_order_is_valid = False

    # Determine the numerical interval allowed for each abundance.
    #
    # Example:
    # abundance 100 -> suffix 1
    # abundance 90  -> suffixes 2-4 when three features are tied
    # abundance 80  -> suffix 5
    abundance_intervals = {}

    if group["total_abundance"].notna().all():
        abundance_counts = (
            group["total_abundance"]
            .value_counts()
            .sort_index(ascending=False)
        )

        first_allowed_suffix = 1

        for abundance_value, count in abundance_counts.items():
            last_allowed_suffix = (
                first_allowed_suffix + int(count) - 1
            )

            abundance_intervals[abundance_value] = (
                first_allowed_suffix,
                last_allowed_suffix,
            )

            first_allowed_suffix = (
                last_allowed_suffix + 1
            )

    for row_index, validation_row in group.iterrows():
        suffix = validation_row[
            "observed_numerical_suffix"
        ]
        abundance = validation_row[
            "total_abundance"
        ]

        within_tie_interval = False

        if (
            pd.notna(suffix)
            and pd.notna(abundance)
            and abundance in abundance_intervals
        ):
            minimum_suffix, maximum_suffix = (
                abundance_intervals[abundance]
            )

            within_tie_interval = (
                minimum_suffix
                <= int(suffix)
                <= maximum_suffix
            )

        row_validation.loc[
            row_index,
            "suffix_within_abundance_tie_interval",
        ] = within_tie_interval

        row_validation.loc[
            row_index,
            "group_suffixes_are_consecutive",
        ] = suffixes_are_consecutive

        row_validation.loc[
            row_index,
            "group_abundance_order_is_valid",
        ] = abundance_order_is_valid

    deepest_rank = group[
        "deepest_accepted_rank"
    ].iloc[0]

    deepest_taxon = group[
        "deepest_accepted_taxon"
    ].iloc[0]

    group_results.append(
        {
            "taxon_group_key": group_key,
            "deepest_accepted_rank": deepest_rank,
            "deepest_accepted_taxon": deepest_taxon,
            "number_of_standardized_units": group_size,
            "minimum_total_abundance":
                group["total_abundance"].min(),
            "maximum_total_abundance":
                group["total_abundance"].max(),
            "minimum_observed_suffix":
                group["observed_numerical_suffix"].min(),
            "maximum_observed_suffix":
                group["observed_numerical_suffix"].max(),
            "number_of_unique_suffixes":
                group["observed_numerical_suffix"].nunique(
                    dropna=True
                ),
            "suffixes_are_consecutive":
                suffixes_are_consecutive,
            "abundance_order_is_valid":
                abundance_order_is_valid,
            "all_prefixes_match_deepest_taxon":
                bool_all(
                    group["prefix_matches_deepest_taxon"]
                ),
            "all_suffix_digit_counts_valid":
                bool_all(
                    group["suffix_digit_count_valid"]
                ),
            "all_rows_within_abundance_tie_interval":
                bool_all(
                    row_validation.loc[
                        group.index,
                        "suffix_within_abundance_tie_interval",
                    ]
                ),
        }
    )

group_validation = pd.DataFrame(group_results)


# ============================================================
# 9. FINAL ROW STATUS
# ============================================================

row_validation["assignment_valid"] = (
    row_validation[
        "identifier_is_globally_unique"
    ]
    & row_validation[
        "prefix_matches_deepest_taxon"
    ]
    & row_validation[
        "valid_numerical_suffix"
    ]
    & row_validation[
        "suffix_digit_count_valid"
    ]
    & row_validation[
        "group_suffixes_are_consecutive"
    ]
    & row_validation[
        "group_abundance_order_is_valid"
    ]
    & row_validation[
        "suffix_within_abundance_tie_interval"
    ]
)

row_validation["status"] = row_validation[
    "assignment_valid"
].map(
    {
        True: "PASS",
        False: "FAIL",
    }
)

group_validation["group_assignment_valid"] = (
    group_validation[
        "suffixes_are_consecutive"
    ]
    & group_validation[
        "abundance_order_is_valid"
    ]
    & group_validation[
        "all_prefixes_match_deepest_taxon"
    ]
    & group_validation[
        "all_suffix_digit_counts_valid"
    ]
    & group_validation[
        "all_rows_within_abundance_tie_interval"
    ]
)

group_validation["status"] = group_validation[
    "group_assignment_valid"
].map(
    {
        True: "PASS",
        False: "FAIL",
    }
)


# ============================================================
# 10. SUMMARY
# ============================================================

total_rows = len(row_validation)
passed_rows = int(
    row_validation["assignment_valid"].sum()
)
failed_rows = total_rows - passed_rows

total_groups = len(group_validation)
passed_groups = int(
    group_validation["group_assignment_valid"].sum()
)
failed_groups = total_groups - passed_groups

summary = pd.DataFrame(
    {
        "metric": [
            "final_database_file",
            "standardized_identifier_column",
            "total_abundance_column",
            "sppn_grouping_rule",
            "rows_evaluated",
            "rows_passing_all_checks",
            "rows_failing_one_or_more_checks",
            "row_validation_percentage",
            "taxonomic_groups_evaluated",
            "groups_passing_all_checks",
            "groups_failing_one_or_more_checks",
            "group_validation_percentage",
            "globally_unique_identifiers",
            "prefixes_matching_deepest_taxon",
            "valid_numerical_suffixes",
            "valid_suffix_digit_counts",
            "rows_within_abundance_tie_interval",
        ],
        "value": [
            str(FINAL_DATABASE_FILE),
            standardized_id_column,
            total_abundance_column,
            "case-sensitive sanitized prefix only; taxonomic rank excluded",
            total_rows,
            passed_rows,
            failed_rows,
            (
                100 * passed_rows / total_rows
                if total_rows
                else 0
            ),
            total_groups,
            passed_groups,
            failed_groups,
            (
                100 * passed_groups / total_groups
                if total_groups
                else 0
            ),
            int(
                row_validation[
                    "identifier_is_globally_unique"
                ].sum()
            ),
            int(
                row_validation[
                    "prefix_matches_deepest_taxon"
                ].sum()
            ),
            int(
                row_validation[
                    "valid_numerical_suffix"
                ].sum()
            ),
            int(
                row_validation[
                    "suffix_digit_count_valid"
                ].sum()
            ),
            int(
                row_validation[
                    "suffix_within_abundance_tie_interval"
                ].sum()
            ),
        ],
    }
)

failed_rows_table = row_validation[
    ~row_validation["assignment_valid"]
].copy()

failed_groups_table = group_validation[
    ~group_validation["group_assignment_valid"]
].copy()


# ============================================================
# 11. EXPORT RESULTS
# ============================================================

run_epithet = RUN_EPITHET

summary_file = (
    OUTPUT_FOLDER
    / f"StandardizedIdentifierValidation_{run_epithet}_summary.csv"
)

row_file = (
    OUTPUT_FOLDER
    / f"StandardizedIdentifierValidation_{run_epithet}_rows.csv"
)

group_file = (
    OUTPUT_FOLDER
    / f"StandardizedIdentifierValidation_{run_epithet}_groups.csv"
)

failed_rows_file = (
    OUTPUT_FOLDER
    / f"StandardizedIdentifierValidation_{run_epithet}_failed_rows.csv"
)

failed_groups_file = (
    OUTPUT_FOLDER
    / f"StandardizedIdentifierValidation_{run_epithet}_failed_groups.csv"
)

summary.to_csv(
    summary_file,
    index=False,
)

row_validation.to_csv(
    row_file,
    index=False,
)

group_validation.to_csv(
    group_file,
    index=False,
)

failed_rows_table.to_csv(
    failed_rows_file,
    index=False,
)

failed_groups_table.to_csv(
    failed_groups_file,
    index=False,
)


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print("\nSTANDARDIZED BIODIVERSITY-UNIT IDENTIFIER VALIDATION\n")

print("Summary")
display(summary)

print("\nTaxonomic-group validation")
display(group_validation)

if len(failed_groups_table) > 0:
    print("\nGroups requiring review")
    display(failed_groups_table)

if len(failed_rows_table) > 0:
    print("\nRows requiring review")
    display(failed_rows_table)

print("\nFiles written:")
print(summary_file)
print(row_file)
print(group_file)
print(failed_rows_file)
print(failed_groups_file)


Project directory: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC
Marker mode: ITS
Subset mode: all_eukaryotes
Run epithet: genus_all_eukaryotes_p1
Final Database: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\Final_Database_genus_all_eukaryotes_p1.csv
Validation output folder: C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation

STANDARDIZED BIODIVERSITY-UNIT IDENTIFIER VALIDATION

Summary


,metric,value
0,final_database_file,C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_R...
1,standardized_identifier_column,SPPN
2,total_abundance_column,Total_Abundance
3,sppn_grouping_rule,case-sensitive sanitized prefix only; taxonomi...
4,rows_evaluated,103777
5,rows_passing_all_checks,103777
6,rows_failing_one_or_more_checks,0
7,row_validation_percentage,100.0
8,taxonomic_groups_evaluated,5723
9,groups_passing_all_checks,5723



Taxonomic-group validation


,taxon_group_key,deepest_accepted_rank,deepest_accepted_taxon,number_of_standardized_units,minimum_total_abundance,maximum_total_abundance,minimum_observed_suffix,maximum_observed_suffix,number_of_unique_suffixes,suffixes_are_consecutive,abundance_order_is_valid,all_prefixes_match_deepest_taxon,all_suffix_digit_counts_valid,all_rows_within_abundance_tie_interval,group_assignment_valid,status
0,Penicillium_canescens,species,Penicillium_canescens,1,822991,822991,1,1,1,True,True,True,True,True,True,PASS
1,Russula_olivacea,species,Russula_olivacea,1,523163,523163,1,1,1,True,True,True,True,True,True,PASS
2,Aspergillus_protuberus,species,Aspergillus_protuberus,1,350071,350071,1,1,1,True,True,True,True,True,True,PASS
3,Inocybe_geophylla,species,Inocybe_geophylla,1,309353,309353,1,1,1,True,True,True,True,True,True,PASS
4,Clavulina_floridana,species,Clavulina_floridana,1,275442,275442,1,1,1,True,True,True,True,True,True,PASS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5718,Grammothelopsis_puiggarii,species,Grammothelopsis_puiggarii,1,1,1,1,1,1,True,True,True,True,True,True,PASS
5719,Gomph10B,order,Gomph10B,1,1,1,1,1,1,True,True,True,True,True,True,PASS
5720,Acramoebidae,family,Acramoebidae,1,1,1,1,1,1,True,True,True,True,True,True,PASS
5721,Glomeraceae_gen10,genus,Glomeraceae_gen10,1,1,1,1,1,1,True,True,True,True,True,True,PASS



Files written:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\StandardizedIdentifierValidation_genus_all_eukaryotes_p1_summary.csv
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\StandardizedIdentifierValidation_genus_all_eukaryotes_p1_rows.csv
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\StandardizedIdentifierValidation_genus_all_eukaryotes_p1_groups.csv
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\StandardizedIdentifierValidation_genus_all_eukaryotes_p1_failed_rows.csv
C:\Users\berna\Desktop\PAPER METADIV\V1_7_11_RUNS\MetaDiv_Builder_V1_7_11_MXA_MXBC\output\ITS\FINAL_DB\validation\StandardizedIdentifierValidation_genus_all_eukaryotes_p1_failed_groups.csv
